# MADRL CityLearn v3 — Tutorial Completo (Google Colab · campaña nueva)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/fix/madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb)

**Proyecto:** Multi-Agente de Aprendizaje por Refuerzo Profundo para gestion coordinada
de flexibilidad energetica, emisiones de CO2 y eficiencia economica en comunidades inteligentes.

**Caso de estudio:** 17 edificios de Iquitos, Peru · dataset congelado `data/campaigns/iquitos_v1/` ·
splits `train_2023` (8 760 h) / `validation_2024` (8 784 h) / `test_2025` (8 760 h).

| Parametro | Valor |
|---|---|
| Algoritmos | HAPPO · MASAC · MATD3 · MAAC |
| Escenarios | E1 (Flexibilidad / OE1) · E2 (CO2 / OE2) · E3 (Costos / OE3) |
| Campaña | `fresh_iquitos_20261101_v1` (alias `iquitos_v1`) · `configs/experiments/iquitos.yaml` |
| Lanzamiento | unica via: `python -m uc3m.application.training.cli` |
| Etapas | `prepare` → `pilot` → `hpo` → congelado → `confirm` → `sensitivity` |
| Piloto | 4 algoritmos · E1 · semilla 901 · 2 episodios x 168 pasos |
| HPO | rejilla `configs/algorithms/native.yaml` · semillas de desarrollo 901-903 · 10 x 8 760 pasos · seleccion en `validation_2024` |
| Confirmacion | 4 algoritmos x 3 escenarios x 12 semillas 1001-1012 · 50 x 8 760 pasos |
| Sensibilidad | dominancia del eje principal p = 0.55 / 0.65 (el centro 0.60 es la confirmacion) · semillas 1001-1003 |
| Intentos | siempre desde el episodio 0; un intento interrumpido queda como fallido y se lanza otro |
| KPIs | `configs/evaluation/kpis.yaml`, calculados desde trazas fisicas; evaluacion congelada en `test_2025` frente a RBC |
| Estadistica | unidad = semilla de entrenamiento; IC 95 % bootstrap (10 000 remuestreos, semilla 7001); Holm |

> **Estado:** este cuaderno no trae resultados. Las secciones 7-10 leen solo la campaña nueva y lo dicen
> explicitamente cuando aun no se ha entrenado nada.

### Fuentes cientificas y de tesis usadas para el diseno

- CityLearn estandariza la evaluacion RL/MARL para demanda respuesta urbana: https://arxiv.org/abs/2012.10504
- CityLearn v2 y CityLearn Challenge documentan KPIs de flexibilidad, carbono y costo: https://escholarship.org/content/qt5t48x8xk/qt5t48x8xk.pdf y https://proceedings.mlr.press/v220/nweye23a.html
- HAPPO/HATRPO justifica actualizacion secuencial y trust region en MARL: https://openreview.net/forum?id=EcGGFkNTxdJ
- MAAC usa criticos centralizados con atencion para escalar agentes: https://proceedings.mlr.press/v97/iqbal19a.html
- MATD3 reduce sobreestimacion mediante doble critico centralizado: https://arxiv.org/abs/1910.01465
- MASAC se apoya en SAC y mezcla QMIX/CTDE: https://arxiv.org/abs/1812.05905 y https://arxiv.org/abs/1803.11485
- PyTorch CUDA y reproducibilidad: https://docs.pytorch.org/docs/stable/notes/cuda.html y https://docs.pytorch.org/docs/stable/notes/randomness.html
- Colab no garantiza tipo de GPU ni duracion; por eso cada intento es independiente y se relanza desde cero: https://research.google.com/colaboratory/faq.html
- Tesis consultadas sobre RL/MARL energetico: Ross May PhD Dalarna 2023, Oxford residential flexibility thesis, Politecnico di Torino MARL building-energy thesis.

## Guia Rapida de Lanzamiento

> **Unica via de lanzamiento:** la CLI de la campaña nueva. Sin `--execute` nada se entrena; con `--execute`
> solo corren las etapas que `configs/experiments/iquitos.yaml` autoriza en `resource_authorization.authorized_stages`.
> Guia operativa completa: `docs/architecture/runbook.md`.

### Mapa de navegacion (orden obligatorio)

| Bloque | Celdas | Funcion |
|--------|--------|---------|
| Setup | **0.verify** → **1.1–1.5** | Runtime, clonado con submodulos fijados, venv 3.9, imports, Drive |
| Campaña | **2.1** → **2.1b** | Configuracion de la campaña y estado de sus etapas |
| Validacion | **3.1** → **4.1** → **5.1** | Dataset congelado, entorno Dec-POMDP, pesos de recompensa |
| Protocolo | **6.1** → **6.2** | Rejilla de hiperparametros, presupuesto y plan piloto |
| Lanzamiento | **7.0** → **7.1** → **7.2** | `--stage prepare`, luego una etapa (opt-in `EXECUTE_STAGE`) |
| Post | **7.3–7.7**, **8.x**, **9.x**, **10** | Estado, auditoria de intentos, KPIs, estadistica y aceptacion |

**Secuencia minima:**
```
0.verify -> 1.1 -> 1.2 -> 1.2b -> 1.3 -> 1.4 -> 1.5
-> 2.1 -> 2.1b -> 6.1 -> 7.0 -> 7.1 (prepare) -> 7.2 (pilot, sin --execute)
```

**Comandos equivalentes en una terminal:**
```
python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage prepare
python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage pilot
python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage pilot --execute
```

### Si Colab se desconecta

No se reanuda nada. El intento interrumpido queda en su carpeta `try_<n>` como evidencia fallida;
relanzar la misma etapa (celda **7.2**) crea un intento nuevo desde el episodio 0 con la misma identidad
de tratamiento. Nunca se concatenan trayectorias.

### Que se crea

```
outputs/campaigns/iquitos_v1/
  campaign_manifest.json        identidad, hashes de entradas, presupuesto
  campaign.json                 reclamo de propiedad de la campaña
  stages/<etapa>.json           tratamientos planificados (con sha256 en el manifiesto)
  stages/<etapa>_result.json    resultado de una etapa ejecutada
  <etapa>/<algoritmo>/<escenario>/<candidato>/seed_<s>/try_<n>/
      attempt.json  child.log  learning_evidence.json  supervision.json  completion.json
```

## Paso 0: Conectar VS Code al runtime A100/H100 de Google Colab

> Haz este paso **UNA SOLA VEZ** antes de ejecutar cualquier celda.
> No se necesita ngrok ni tunnels: la extension `google.colab` de VS Code
> maneja la conexion directamente con tu cuenta de Google.

---

### 0.1  Seleccionar el kernel Colab en VS Code

1. Abre este notebook en VS Code
   (`examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb`)
2. Haz clic en **"Select Kernel"** (esquina superior derecha del notebook)
3. En la lista elige **"Google Colab"**
4. Si pide autenticacion → cuenta Google con Colab Pro+ y espacio en Drive
5. Runtime: **"New Colab Server"** → acelerador **A100** (o H100 si disponible) → forma **High-RAM**
   *(la etapa `pilot` mide RAM/VRAM/tiempo reales antes de autorizar etapas largas)*

> Si no ves "Google Colab": `Ctrl+Shift+P` → **"Colab: Sign In"**, autentica, repite.

---

### 0.2  Verificar la conexion

Ejecuta la celda de codigo siguiente (**0.verify**). Muestra Python, GPU y RAM del runtime.
Si no aparece GPU en Colab → vuelve al paso 0.1.

---

### 0.3  Flujo de trabajo diario

```
VS Code (editor local)
       │
       │  google.colab extension
       ▼
Colab A100/H100 runtime
  /content/MADRLCitytleranflexresdr/   ← repo clonado en celda 1.2
  /content/drive/MyDrive/MADRL_campaigns/iquitos_v1/  ← copia de los intentos
```

- El **codigo se ejecuta en el runtime Colab**.
- Los **outputs y graficas** aparecen en VS Code.
- Si Colab desconecta: repetir 0.1 y relanzar la etapa en **7.2**; el intento interrumpido queda como fallido.

> **Apertura directa en Colab (sin VS Code):** badge del titulo o
> [`madrl_citylearn_v3_tutorial.ipynb` en GitHub](https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/fix/madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb)

In [ ]:
# ── 0.verify  Verificar conexion al runtime (Colab o local) ────────────────────
import importlib.util
import platform
import subprocess
import sys

IN_COLAB = importlib.util.find_spec('google.colab') is not None

print(f"Entorno  : {'Google Colab' if IN_COLAB else 'local'} ({platform.system()})")
print(f"Python   : {sys.version.split()[0]} ({sys.executable})")

# GPU: informativa. La etapa pilot mide los recursos reales antes de autorizar etapas largas.
try:
    gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                                  text=True, stderr=subprocess.DEVNULL).strip()
    print(f"GPU      : {gpu}")
except Exception:
    print('GPU      : nvidia-smi no disponible' + (' — selecciona un runtime con GPU' if IN_COLAB else ''))

# RAM total del runtime.
try:
    import psutil
    print(f"RAM      : {psutil.virtual_memory().total / 1024**3:.1f} GiB")
except ImportError:
    print('RAM      : psutil no disponible (se instala en la celda 1.3)')

## Sección 0: Arquitectura del Proyecto — 9 Diagramas de Defensa

> Arquitectura del experimento MADRL CityLearn v3 con la campaña nueva.
> Renderizado: **mermaid.ink** (SVG estático) con fallback **Mermaid@10 CDN**. Ejecutar primero la celda **0.0**.

| Celda | Diagrama |
|-------|----------|
| 0.0 | Helper Mermaid + carga CDN |
| 0.1 | Visión general (inicio → resultado) |
| 0.2 | Pipeline del dataset de la campaña `iquitos_v1` |
| 0.3 | Arquitectura Dec-POMDP y CTDE — 17 agentes |
| 0.4 | Los 4 algoritmos MADRL: taxonomía |
| 0.5 | Flujo de entrenamiento: etapas prepare → pilot → hpo → confirm → sensitivity |
| 0.6 | Recompensa multiobjetivo por escenario |
| 0.7 | Evaluación congelada y decisión sin ganador forzado |
| 0.8 | Infraestructura: local para preparar, Colab para entrenar |
| 0.9 | Estructura de capas del repositorio |

In [ ]:
# ── 0.0  Helper Mermaid — renderiza los 9 diagramas de arquitectura ──────────
# Estrategia: mermaid.ink API (SVG estatico guardado en notebook) con fallback CDN
import json
import base64
import urllib.request
from IPython.display import display, HTML

_diagram_idx = [0]

def render_mermaid(title, code, height=520):
    """Renderiza diagrama Mermaid via mermaid.ink (estatico) o CDN (fallback)."""
    _diagram_idx[0] += 1
    uid = f"mmd_{_diagram_idx[0]}"

    # ── Intento 1: mermaid.ink API → SVG embebido en el output (offline despues) ──
    try:
        encoded = base64.urlsafe_b64encode(code.strip().encode("utf-8")).decode("utf-8")
        url = f"https://mermaid.ink/svg/{encoded}"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            svg = resp.read().decode("utf-8")
        html = f"""<div style="margin:16px 0;border:1px solid #e2e8f0;border-radius:10px;
padding:20px;background:#f8fafc;font-family:sans-serif;">
  <h4 style="margin:0 0 14px 0;color:#0f172a;font-size:14px;">{title}</h4>
  <div style="overflow:auto;max-height:{height + 80}px;">{svg}</div>
</div>"""
        display(HTML(html))
        return
    except Exception as _e:
        pass  # fallback below

    # ── Intento 2: CDN Mermaid@10 (requiere JS habilitado en el navegador) ──────
    code_js = json.dumps(code, ensure_ascii=False)
    html = f"""<div style="margin:16px 0;border:1px solid #e2e8f0;border-radius:10px;
padding:20px;background:#f8fafc;font-family:sans-serif;">
  <h4 style="margin:0 0 14px 0;color:#0f172a;font-size:14px;">{title}</h4>
  <div id="{uid}" style="min-height:{height}px;"></div>
  <script>
  (function(){{
    var el=document.getElementById("{uid}");
    el.textContent={code_js};
    el.className="mermaid";
    function tryR(){{
      if(window._mermaidReady&&typeof mermaid!=="undefined"){{
        try{{mermaid.run({{nodes:[el]}});}}catch(e){{console.error(e);}}
      }}else{{
        if(!window._mermaidCDNLoading){{
          window._mermaidCDNLoading=true;
          var s=document.createElement("script");
          s.src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
          s.onload=function(){{mermaid.initialize({{startOnLoad:false,theme:"default",securityLevel:"loose"}});window._mermaidReady=true;}};
          document.head.appendChild(s);
        }}
        setTimeout(tryR,400);
      }}
    }}
    tryR();
  }})();
  </script>
</div>"""
    display(HTML(html))

print("✅  Helper Mermaid listo (mermaid.ink + CDN fallback). Ejecuta celdas 0.1-0.9.")


In [ ]:
# ── 0.1  Diagrama 1 ────────────────────────────────────────
render_mermaid("Diagrama 1 — Vision General del Proyecto (inicio a fin)", r"""
flowchart LR
    subgraph ORIGEN["① Origen del proyecto"]
        direction TB
        PROB(["Problema de investigacion\n¿Que MADRL gestiona mejor\nflexibilidad + CO2 + costos\nen comunidades inteligentes?"])
        OBJ["Objetivos especificos\nOE1 Flexibilidad\nOE2 Emisiones CO2\nOE3 Costos energeticos"]
        PROB --> OBJ
    end

    subgraph DATOS["② Dataset de la campaña nueva"]
        direction TB
        RAW["Fuentes primarias auditadas\nCityLearn/data/buildingcsv/\n17 edificios"]
        PIPE["Construccion + splits\ntools/dataset/build_fresh_campaign_dataset.py\ntools/dataset/split_campaign.py"]
        DS[("data/campaigns/iquitos_v1/\ntrain_2023 · validation_2024 · test_2025\n8760 / 8784 / 8760 h")]
        RAW --> PIPE --> DS
    end

    subgraph SIM["③ Simulador"]
        direction TB
        V2["CityLearn v2 base\nFisica edificios\nBESS + PV + EV"]
        V3["CityLearn v3\nDec-POMDP 17 agentes\nCTDE: critico centralizado\nRecompensa por ejes"]
        V2 -->|"se extiende con\ncapa experimental"| V3
    end

    subgraph MADRL_BLOQUE["④ Campaña de entrenamiento"]
        direction TB
        ALGS["4 algoritmos\nHAPPO · MASAC\nMATD3 · MAAC"]
        EJES["3 escenarios\nE1 Flex · E2 CO2 · E3 Costo"]
        CLI["python -m uc3m.application.training.cli\npilot → hpo → confirm → sensitivity\ncada intento desde cero"]
        ALGS --> EJES --> CLI
    end

    subgraph EVAL["⑤ Evaluacion congelada"]
        direction TB
        KPIS["KPIs desde trazas fisicas\nconfigs/evaluation/kpis.yaml\nprueba test_2025 vs RBC"]
        STAT["Estadistica por semilla\nIC 95% bootstrap · Holm\nn = 12 semillas"]
        KPIS --> STAT
    end

    subgraph RESULT["⑥ Resultado"]
        direction TB
        DEC(["Decision sin ganador forzado\nun candidato, ninguno unico\no estudio negativo completo"])
        TESIS["Entrega con hashes\ntools/eval/report_fresh_campaign.py"]
        DEC --> TESIS
    end

    ORIGEN --> DATOS
    DATOS --> SIM
    SIM --> MADRL_BLOQUE
    MADRL_BLOQUE --> EVAL
    EVAL --> RESULT

    classDef origen fill:#fef3c7,stroke:#d97706,color:#1c1917,stroke-width:2px
    classDef datos fill:#dbeafe,stroke:#2563eb,color:#1c1917,stroke-width:2px
    classDef sim fill:#e0e7ff,stroke:#4f46e5,color:#1c1917,stroke-width:2px
    classDef madrl fill:#fae8ff,stroke:#a21caf,color:#1c1917,stroke-width:2px
    classDef eval fill:#dcfce7,stroke:#16a34a,color:#1c1917,stroke-width:2px
    classDef result fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px

    class ORIGEN origen
    class DATOS datos
    class SIM sim
    class MADRL_BLOQUE madrl
    class EVAL eval
    class RESULT result
""", height=520)

In [ ]:
# ── 0.2  Diagrama 2 ────────────────────────────────────────
render_mermaid("Diagrama 2 — Pipeline del Dataset de la Campaña (iquitos_v1)", r"""
flowchart TD
    subgraph INSUMOS["Fuentes primarias aprobadas"]
        FAC["Facturas y perfiles por edificio\nCityLearn/data/buildingcsv/"]
        RAWC["Copia congelada de entradas\ndata/campaigns/iquitos_v1/raw/"]
        FAC --> RAWC
    end

    subgraph BUILD["Construccion (T2)"]
        GEN["tools/dataset/build_fresh_campaign_dataset.py\nprocedencia + balance mensual\nquality_flags.csv"]
        ACC["dataset_acceptance.json\ndataset_verified · dataset_hash"]
        GEN --> ACC
    end

    subgraph SPLIT["Splits y normalizador congelados (T3)"]
        SPL["tools/dataset/split_campaign.py\nsplit_manifest.json"]
        TR["train_2023\n8760 h"]
        VA["validation_2024\n8784 h"]
        TE["test_2025\n8760 h"]
        NORM["normalizer.json\nlimites ajustados solo con 2023"]
        SPL --> TR & VA & TE
        TR --> NORM
    end

    RAWC --> GEN
    ACC --> SPL

    classDef ins fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef pipe fill:#e0f2fe,stroke:#0284c7,color:#1c1917
    classDef ds fill:#dbeafe,stroke:#2563eb,color:#1c1917
    class INSUMOS ins
    class BUILD pipe
    class SPLIT ds
""", height=620)

In [ ]:
# ── 0.3  Diagrama 3 ────────────────────────────────────────
render_mermaid("Diagrama 3 — Arquitectura Dec-POMDP y CTDE de los 17 Agentes", r"""
flowchart TD
    subgraph ENV["Entorno CityLearn v3 (simulacion horaria)"]
        direction LR
        B1["Edificio 1\nBESS + PV + EV"]
        B2["Edificio 2\nBESS + PV + EV"]
        BN["... Edificio 17\nBESS + PV + EV"]
        GRID(["Red electrica\nElectro Oriente\nSistema aislado diesel"])
        B1 --- B2 --- BN
        B1 & B2 & BN --> GRID
    end

    subgraph OBS["Observaciones locales oᵢ(t) — heterogeneas por edificio"]
        direction LR
        TIME["Tiempo\nmes hora tipo_dia"]
        PHYS["Fisica edificio\ncarga_no_desplazable\ngeneracion_solar"]
        BESS_OBS["Estado BESS\nSOC accion_previa"]
        EV_OBS["Estado EV\nSOC_k salida_k\nSOC_req_k llegada_k"]
        SIG["Senales globales\ncarbono precio\nclima"]
    end

    subgraph POLICY["Politicas descentralizadas (ejecucion)"]
        P1["π₁(a₁|o₁)\nred neuronal\nedificio 1"]
        P2["π₂(a₂|o₂)\nred neuronal\nedificio 2"]
        PN["π₁₇(a₁₇|o₁₇)\nred neuronal\nedificio 17"]
    end

    subgraph ACTIONS["Acciones locales aᵢ(t)"]
        A1["a₁: BESS carga/descarga\nEV carga\nLavadora on/off"]
        A2["a₂: BESS · EV · Lavadora"]
        AN["a₁₇: BESS · EV · Lavadora"]
    end

    subgraph CRITIC["Critico centralizado (solo en entrenamiento CTDE)"]
        STATE["Estado global s = concat(o₁,...,o₁₇)\nV(s) o Q(s,a) centralizado"]
        REWARD["Recompensa de equipo\nreward_aggregation = team_mean\nmismo pago comun para los 17 agentes"]
        STATE --> REWARD
    end

    subgraph UPDATE["Actualizacion de politicas (CTDE)"]
        GRAD["Gradiente con informacion global\nHAPPO: secuencial con trust region\nMASAC: Q-mix + SAC discreto\nMATD3: TD3 con critico centralizado\nMAAC: attention sobre Q"]
    end

    ENV -->|"emite oᵢ(t)"| OBS
    OBS --> POLICY
    POLICY -->|"accion aᵢ"| ACTIONS
    ACTIONS -->|"aplica en entorno"| ENV
    ENV -->|"estado global\nsolo entrenamiento"| CRITIC
    CRITIC --> UPDATE
    UPDATE -->|"actualiza pesos"| POLICY

    classDef env fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef obs fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef pol fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef crit fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef act fill:#ffedd5,stroke:#ea580c,color:#1c1917
    class ENV env
    class OBS obs
    class POLICY,ACTIONS pol
    class CRITIC,UPDATE crit
""", height=720)

In [ ]:
# ── 0.4  Diagrama 4 ────────────────────────────────────────
render_mermaid("Diagrama 4 — Los 4 Algoritmos MADRL: Taxonomia y Diferencias", r"""
flowchart LR
    subgraph HAPPO_BOX["HAPPO — Heterogeneous-Agent PPO"]
        direction TB
        HAPPO_T["Tipo: On-policy\nActualizacion secuencial\nTrust region por agente"]
        HAPPO_C["Critico: Centralizado V(s)\nActor: π(a|o) local\nBackend: external/HARL"]
    end

    subgraph MASAC_BOX["MASAC — Multi-Agent SAC Discreto"]
        direction TB
        MASAC_T["Tipo: Off-policy\nEntropy regularization\nAcciones discretas por eje"]
        MASAC_C["Critico: Q-mix centralizado\nActor: π(a|o) + temperatura\nBackend: external/MARL/src"]
    end

    subgraph MATD3_BOX["MATD3 — Multi-Agent TD3"]
        direction TB
        MATD3_T["Tipo: Off-policy\nDoble critico (anti-overest)\nPolicy delay + target noise"]
        MATD3_C["Critico: Par Q₁ Q₂ centralizado\nActor: μ(o) deterministico\nBackend: external/off-policy"]
    end

    subgraph MAAC_BOX["MAAC — Multi-Agent Attention Critic"]
        direction TB
        MAAC_T["Tipo: Off-policy\nAtencion sobre agentes\nSAC con Q de atencion"]
        MAAC_C["Critico: Attention SAC Q(s,a)\nActor: π(a|o) estocastico\nBackend: external/MAAC"]
    end

    HPS["Hiperparametros\nconfigs/algorithms/native.yaml\nrejilla lr · hidden · gamma\nseleccion en validation_2024"]

    HAPPO_BOX --> HPS
    MASAC_BOX --> HPS
    MATD3_BOX --> HPS
    MAAC_BOX --> HPS

    HPS -->|"configuracion congelada"| COMP(["Confirmacion\n12 semillas 1001-1012\nevaluacion congelada test_2025"])

    classDef happo fill:#dbeafe,stroke:#2563eb,color:#1c1917,stroke-width:2px
    classDef masac fill:#fae8ff,stroke:#a21caf,color:#1c1917,stroke-width:2px
    classDef matd3 fill:#dcfce7,stroke:#16a34a,color:#1c1917,stroke-width:2px
    classDef maac fill:#fef3c7,stroke:#d97706,color:#1c1917,stroke-width:2px
    class HAPPO_BOX happo
    class MASAC_BOX masac
    class MATD3_BOX matd3
    class MAAC_BOX maac
""", height=600)

In [ ]:
# ── 0.5  Diagrama 5 ────────────────────────────────────────
render_mermaid("Diagrama 5 — Flujo de Entrenamiento: Etapas de la Campaña Nueva", r"""
flowchart TD
    START(["python -m uc3m.application.training.cli\n--config configs/experiments/iquitos.yaml --stage <etapa>\nsin opcion execute: solo plan · con execute: solo etapas autorizadas"])

    PREP["prepare\ncampaign_manifest.json + stages/*.json\npresupuesto de pasos · no entrena"]
    PILOT["pilot\n4 algoritmos · E1 · semilla 901\n2 episodios x 168 pasos · mide RAM/VRAM/tiempo"]
    HPO["hpo\nrejilla native.yaml · semillas 901-903\n10 episodios x 8760 pasos\nseleccion en validation_2024"]
    FREEZE["configuration_freezes.json\nconfiguracion congelada por algoritmo"]
    CONF["confirm\n4 algoritmos x 3 escenarios\n12 semillas 1001-1012 · 50 x 8760 pasos"]
    SENS["sensitivity\ndominancia p = 0.55 / 0.65\nsemillas 1001-1003 · catalogos derivados con hash"]

    START --> PREP --> PILOT --> HPO --> FREEZE --> CONF --> SENS

    subgraph ATTEMPT["Cada intento (desde el episodio 0)"]
        direction LR
        PATH["outputs/campaigns/iquitos_v1/<etapa>/<algoritmo>/<escenario>/<candidato>/seed_<s>/try_<n>/"]
        FILES["attempt.json · child.log\nlearning_evidence.json · supervision.json\ncompletion.json"]
        PATH --- FILES
    end

    PILOT & HPO & CONF & SENS -->|"intento fallido se conserva;\nrelanzar crea try nuevo"| ATTEMPT

    classDef stage fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef art fill:#f8fafc,stroke:#64748b,color:#1c1917
    class PREP,PILOT,HPO,FREEZE,CONF,SENS stage
    class ATTEMPT art
""", height=760)

In [ ]:
# ── 0.6  Diagrama 6 ────────────────────────────────────────
render_mermaid("Diagrama 6 — Recompensa Multiobjetivo por Escenario", r"""
flowchart LR
    subgraph REW_FUNC["CityLearnV3MADRLRewardFunction (perfil fresh_kpi_aligned_v5)"]
        direction TB
        COMP1["Eje FLEX\npico convexo + rampa lineal\nsobre la potencia neta coincidente del distrito"]
        COMP2["Eje CO2\nimportacion del edificio\n* intensidad de carbono"]
        COMP3["Eje COSTO\nimportacion del edificio\n* precio (sin credito de exportacion)"]
    end

    subgraph PENAL["Penalizaciones fuera del vector ponderado"]
        direction TB
        EV["Servicio / violacion EV"]
        BESS["Ciclado BESS"]
    end

    subgraph PESOS["Pesos por escenario (configs/objective_policies.yaml)"]
        direction TB
        PE1["E1 Flexibilidad\nflex=0.60 co2=0.15 cost=0.25"]
        PE2["E2 CO2\nflex=0.25 co2=0.60 cost=0.15"]
        PE3["E3 Costos\nflex=0.15 co2=0.25 cost=0.60"]
    end

    subgraph TEAM["Agregacion de equipo"]
        MEAN["reward_aggregation = team_mean\npago comun para los 17 agentes"]
    end

    REW_FUNC --> PESOS
    PESOS --> TEAM
    PENAL --> TEAM
    TEAM --> SUP(["Supervision por intento\nsupervision.json: se rechaza si EV/BESS\ndominan los ejes ponderados"])

    classDef rw fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef pe fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef mx fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef pf fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    class REW_FUNC rw
    class PESOS pe
    class TEAM mx
    class PENAL pf
""", height=600)

In [ ]:
# ── 0.7  Diagrama 7 ────────────────────────────────────────
render_mermaid("Diagrama 7 — Evaluacion Congelada y Decision sin Ganador Forzado", r"""
flowchart TD
    subgraph IN["Entrada: intentos verificados de la campaña"]
        direction LR
        LE["learning_evidence.json\nactualizaciones reales + checkpoint recargable"]
        RES["stages/<etapa>_result.json\ntratamientos completos"]
    end

    subgraph KPI["KPIs desde trazas fisicas por paso"]
        direction TB
        CAT["configs/evaluation/kpis.yaml\nprimarios: pico diario, rampa,\n1 - factor de carga, CO2 total, costo total"]
        BASE["Linea base RBC\nuc3m/application/evaluation/baselines.py"]
        GAIN["Ganancia por eje vs RBC\nen test_2025 (validation_2024 solo para seleccion)"]
        CAT --> GAIN
        BASE --> GAIN
    end

    subgraph STAT["Estadistica por semilla (fresh_analysis)"]
        direction TB
        CI["IC 95% bootstrap\n10000 remuestreos · semilla 7001"]
        FAM["Familias separadas\nomnibus · pares · impacto vs RBC\ncorreccion de Holm"]
        HV["Hipervolumen con referencia comun\nTOPSIS con pesos congelados"]
        CI --> FAM --> HV
    end

    subgraph DEC["Decision"]
        D1(["candidato unico\no no_unique_candidate\no estudio negativo completo"])
    end

    IN --> KPI --> STAT --> DEC

    classDef inp fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef kpi fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef stat fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef res fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px
    class IN inp
    class KPI kpi
    class STAT stat
    class DEC res
""", height=700)

In [ ]:
# ── 0.8  Diagrama 8 ────────────────────────────────────────
render_mermaid("Diagrama 8 — Infraestructura: Local para preparar, Colab para entrenar", r"""
flowchart LR
    subgraph DEV["Desarrollo local (Windows)"]
        direction TB
        CODE["Codigo fuente\nuc3m/ · configs/ · tools/\nCityLearn/ y external/ protegidos"]
        VENV["Entorno .venv39-citylearn-v3"]
        PREP["CLI: etapa prepare\nmanifiestos + presupuesto"]
        CODE --> VENV --> PREP
    end

    subgraph GIT["Repositorio GitHub"]
        REPO_P["Mac-Tapia/MADRLCitytleranflexresdr\nsubmodulos en commits fijados"]
    end

    subgraph COLAB["Google Colab (entrenamiento)"]
        direction TB
        NB["examples_madrl_v3/madrl_citylearn_v3_modular_colab.ipynb\no este tutorial"]
        RUN["CLI: etapa con execute\nsolo etapas autorizadas"]
        DRIVE["Google Drive\nMADRL_campaigns/iquitos_v1/"]
        NB --> RUN --> DRIVE
    end

    DEV -->|"git push"| GIT
    GIT -->|"clone --recursive"| COLAB
    COLAB -->|"intentos copiados"| DEV

    classDef colab fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef dev fill:#e0f2fe,stroke:#0284c7,color:#1c1917
    classDef git fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    class COLAB colab
    class DEV dev
    class GIT git
""", height=560)

In [ ]:
# ── 0.9  Diagrama 9 ────────────────────────────────────────
render_mermaid("Diagrama 9 — Estructura de Capas del Software", r"""
flowchart TD
    subgraph L1["Capa 1: Simulador base (CityLearn v2)"]
        direction LR
        CL2["CityLearn/citylearn/\nFisica edificios + BESS + PV + EV"]
    end

    subgraph L2["Capa 2: Extension experimental (CityLearn v3)"]
        direction LR
        ENV3["CityLearn/citylearn/v3/\nDec-POMDP · objetivos · configuracion"]
    end

    subgraph L3["Capa 3: Dominio y recompensa del proyecto"]
        direction LR
        DOM["uc3m/domain/campaigns.py\nidentidad y hash de campaña"]
        REW["uc3m/reward/\nejes y telemetria de terminos"]
        DOM --- REW
    end

    subgraph L4["Capa 4: Backends MADRL externos"]
        direction LR
        HARL["external/HARL/\nHAPPO"]
        MARL["external/MARL/src/\nMASAC"]
        OFFP["external/off-policy/\nMATD3"]
        MAAC_B["external/MAAC/\nMAAC"]
        HARL --- MARL --- OFFP --- MAAC_B
    end

    subgraph L5["Capa 5: Lanzamiento unico de la campaña"]
        direction LR
        CLI["uc3m/application/training/cli.py\nunico punto de entrada"]
        CAMP["uc3m/application/training/fresh_campaign.py\nDAG · etapas · intentos desde cero"]
        NAT["uc3m/adapters/algorithms/native.py\npuente al backend nativo"]
        CLI --- CAMP --- NAT
    end

    subgraph L6["Capa 6: Evaluacion, aceptacion y entrega"]
        direction LR
        KPI["uc3m/application/evaluation/kpi_contract.py\nKPIs desde trazas"]
        AN["uc3m/application/evaluation/fresh_analysis.py\nestadistica por semilla"]
        ACC["uc3m/application/training/fresh_acceptance.py\npuerta de aceptacion"]
        KPI --- AN --- ACC
    end

    L1 -->|"extiende"| L2
    L2 -->|"usa"| L3
    L3 -->|"conecta"| L4
    L4 -->|"invocado solo por"| L5
    L5 -->|"genera intentos para"| L6

    classDef l1 fill:#f1f5f9,stroke:#64748b,color:#1c1917
    classDef l2 fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef l3 fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef l4 fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef l5 fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef l6 fill:#dcfce7,stroke:#16a34a,color:#1c1917
    class L1 l1
    class L2 l2
    class L3 l3
    class L4 l4
    class L5 l5
    class L6 l6
""", height=700)

## Sección 1: Configuración inicial

Celdas **1.1–1.5**: GPU/CUDA, clonado del proyecto con submódulos en los commits fijados por el padre, venv Python 3.9, `sys.path`/imports y montaje opcional de Google Drive. Obligatorias antes de **2.1**.

In [ ]:
# ── 1.1  Verificar entorno: IN_COLAB, GPU, CUDA, Python 3.9 ─────────────────
import sys
import subprocess
import importlib.util

# ── Deteccion automatica de entorno ──────────────────────────────────────────
IN_COLAB = importlib.util.find_spec('google.colab') is not None

print(f"Ejecutando en Google Colab : {IN_COLAB}")
print(f"Python version             : {sys.version.split()[0]}")
print(f"Plataforma                 : {sys.platform}")

if not sys.version_info[:2] == (3, 9):
    msg = (
        f"Python {sys.version.split()[0]} detectado; el proyecto usa Python 3.9. "
        "La celda 1.3 crea/usa .venv39-citylearn-v3 y la CLI se ejecuta con ese interprete."
    )
    print(f"[{'WARN' if IN_COLAB else 'INFO'}] {msg}")

# ── GPU via nvidia-smi ───────────────────────────────────────────────────────
res = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(f"GPU                        : {res.stdout.strip() if res.returncode == 0 else 'nvidia-smi no disponible'}")

# ── Verificacion PyTorch + CUDA ───────────────────────────────────────────────
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch version            : {torch.__version__}")
    print(f"CUDA disponible            : {cuda_ok}")
    if cuda_ok:
        name = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"Dispositivo GPU            : {name}")
        print(f"VRAM total                 : {mem:.1f} GiB")
        print("La etapa pilot mide RAM/VRAM/tiempo reales; no se fija aqui un umbral de hardware.")
    elif IN_COLAB:
        print("[WARN] CUDA no disponible en Colab: selecciona un runtime con GPU antes de ejecutar etapas.")
    else:
        print("[INFO] CUDA no disponible — entorno local en CPU (suficiente para la etapa prepare).")
except ImportError:
    print("[INFO] torch no disponible en el kernel. La celda 1.3 lo instala en .venv39-citylearn-v3.")

In [ ]:
# ── 1.2  Clonar el proyecto con sus submodulos fijados ──────────────────────
# El padre fija el commit de CityLearn y de external/*; no se mueven a otras ramas,
# porque la identidad de la campaña incluye un hash del codigo fuente.
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'
REPO = '/content/MADRLCitytleranflexresdr'

if IN_COLAB and not os.path.exists(f'{REPO}/.git'):
    if os.path.exists(REPO):
        raise RuntimeError(f'{REPO} existe pero sin .git. Elimina la carpeta y vuelve a ejecutar.')
    subprocess.check_call(['git', 'clone', '--recursive', REPO_URL, REPO])
elif not IN_COLAB:
    # En local se usa el checkout actual: la raiz es la carpeta con uc3m/ y configs/experiments/.
    _here = Path.cwd().resolve()
    REPO = str(next(p for p in (_here, *_here.parents)
                    if (p / 'uc3m').is_dir() and (p / 'configs' / 'experiments').is_dir()))

# En Colab asegura los submodulos en los commits que fija el padre (en local no se tocan).
if IN_COLAB:
    subprocess.check_call(['git', 'submodule', 'update', '--init', '--recursive'], cwd=REPO)

origin = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=REPO, text=True).strip()
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('REPO   =', REPO)
print('origin =', origin)
print('HEAD   =', head)
if origin.rstrip('/').removesuffix('.git') != REPO_URL.removesuffix('.git'):
    raise RuntimeError(f'Origen inesperado: {origin}')
print(subprocess.check_output(['git', 'submodule', 'status'], cwd=REPO, text=True))

PROJECT_DIR = REPO
os.chdir(REPO)

In [ ]:
# ── 1.2b  Verificar los archivos de la via nueva ─────────────────────────────
from pathlib import Path

ROOT = Path(globals().get('PROJECT_DIR', Path.cwd()))
REQUIRED = [
    'uc3m/application/training/cli.py',
    'uc3m/application/training/fresh_campaign.py',
    'uc3m/application/training/fresh_acceptance.py',
    'configs/experiments/iquitos.yaml',
    'configs/algorithms/native.yaml',
    'configs/scenarios.yaml',
    'configs/objective_policies.yaml',
    'configs/evaluation/kpis.yaml',
    'data/campaigns/iquitos_v1/dataset_acceptance.json',
    'data/campaigns/iquitos_v1/split_manifest.json',
    'data/campaigns/iquitos_v1/normalizer.json',
    'data/campaigns/iquitos_v1/splits/train_2023/schema.json',
    'data/campaigns/iquitos_v1/splits/validation_2024/schema.json',
    'data/campaigns/iquitos_v1/splits/test_2025/schema.json',
]
missing = [path for path in REQUIRED if not (ROOT / path).is_file()]
for path in REQUIRED:
    print('[OK]  ' if path not in missing else '[FALTA]', path)
if missing:
    raise FileNotFoundError(f'Faltan archivos de la campaña nueva: {missing}')
print('\nCelda 1.2b OK: CLI, configuracion y dataset congelado presentes.')

In [ ]:
# 1.3 Instalar dependencias del proyecto de forma reproducible
# Usa Python 3.9 del proyecto. Si Colab entrega kernel 3.11, crea/usa .venv39-citylearn-v3.
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = '/content/MADRLCitytleranflexresdr'
PROJECT_ROOT = Path(PROJECT_DIR)
if not PROJECT_ROOT.exists() and Path.cwd().name == 'MADRLCitytleranflexresdr':
    PROJECT_ROOT = Path.cwd()
    PROJECT_DIR = str(PROJECT_ROOT)

VENV_DIR = PROJECT_ROOT / '.venv39-citylearn-v3'
SETUP_LOG = Path('/tmp/madrl_py39_setup.log')
CONSTRAINTS = Path('/tmp/madrl_compat.txt')
PYTHON_REQUIRED = (3, 9)
PYTHON_MIN = PYTHON_REQUIRED
PYTHON_MAX_EXCLUSIVE = (3, 10)
PYTORCH_INDEX_CU126 = 'https://download.pytorch.org/whl/cu126'
PYTORCH_INDEX_CU128 = 'https://download.pytorch.org/whl/cu128'
TORCH_PACKAGES = ('torch', 'torchvision')

def detect_pytorch_cuda_wheel():
    """Blackwell (sm_120, RTX PRO 6000) requiere wheels cu128; A100/H100 usan cu126."""
    try:
        name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True, stderr=subprocess.DEVNULL,
        ).strip().splitlines()[0].strip()
    except Exception:
        return PYTORCH_INDEX_CU126, None
    upper = name.upper()
    blackwell = any(k in upper for k in ('BLACKWELL', 'RTX PRO 6000', 'RTX 50'))
    index = PYTORCH_INDEX_CU128 if blackwell else PYTORCH_INDEX_CU126
    return index, name

PYTORCH_INDEX_URL, _DETECTED_GPU = detect_pytorch_cuda_wheel()
print(f'[torch] GPU: {_DETECTED_GPU or "(sin nvidia-smi)"} -> index {PYTORCH_INDEX_URL.split("/")[-1]}')

COMPAT_WHEELS = [
    'numpy==1.23.5',
    'pandas==2.0.3',
    'scipy==1.10.1',
    'scikit-learn==1.2.2',
    'matplotlib==3.7.5',
    'seaborn==0.12.2',
]
PINNED = {
    'numpy': '1.23.5',
    'pandas': '2.0.3',
    'scipy': '1.10.1',
    'scikit-learn': '1.2.2',
    'matplotlib': '3.7.5',
    'seaborn': '0.12.2',
    'gymnasium': '0.28.1',
    'pettingzoo': '1.12.0',
}
BASE_DEPS = [
    *COMPAT_WHEELS,
    'pyyaml',
    'requests>=2.28',
    'tqdm>=4.65',
    'psutil>=5.9',
    'platformdirs>=3.0',
    'protobuf==3.20.3',
    'gymnasium==0.28.1',
    'pettingzoo==1.12.0',
    'gym==0.20.0',
    'tensorboard',
    'tensorboardX',
    'setproctitle',
    'simplejson',
    'absl-py',
    'dm-tree',
    'importlib-metadata>=6.0,<9',
]
NO_DEPS_UTILS = [
    'supersuit==3.2.0',
    'icecream==2.1.3',
]
EDITABLES = [
    'CityLearn/',
    'external/HARL/',
    'external/off-policy/',
    'external/MAAC/',
    'external/MARL/src/',
]
BINARY_DEPS = ('numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn')

ABI_CHECK = """
import importlib
import json
import sys

modules = {
    'torch': 'torch',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'gym': 'gym',
    'gymnasium': 'gymnasium',
    'pettingzoo': 'pettingzoo',
    'citylearn.v3.environment': 'citylearn.v3.environment',
}
CRITICAL = {'numpy', 'scipy', 'sklearn', 'gymnasium', 'pettingzoo', 'citylearn.v3.environment'}
versions = {'python': sys.version.split()[0], 'executable': sys.executable}
failures = {}
for label, module_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        versions[label] = getattr(module, '__version__', 'importado')
    except Exception as exc:
        failures[label] = repr(exc)
        versions[label] = f'ERROR: {exc}'
try:
    import torch
    versions['torch_cuda_available'] = bool(torch.cuda.is_available())
    versions['torch_cuda'] = getattr(torch.version, 'cuda', None)
except Exception as exc:
    versions['torch_cuda_error'] = repr(exc)
print(json.dumps(versions, indent=2, sort_keys=True))
critical_failures = {k: v for k, v in failures.items() if k in CRITICAL}
if critical_failures:
    print('CRITICAL_FAILURES: ' + json.dumps(critical_failures), file=sys.stderr)
    sys.exit(1)
elif failures:
    print('NON_CRITICAL_FAILURES: ' + json.dumps(failures), file=sys.stderr)
"""


def write_log(text):
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    with SETUP_LOG.open('a', encoding='utf-8') as f:
        f.write(text)
        if not text.endswith('\n'):
            f.write('\n')


def print_log_tail(lines=80):
    if not SETUP_LOG.exists():
        return
    tail = SETUP_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-lines:]
    print(f'\n[TAIL {SETUP_LOG}]')
    print('\n'.join(tail))


def run(cmd, *, cwd=None, env=None, check=True):
    cmd = [str(part) for part in cmd]
    message = '+ ' + ' '.join(cmd)
    print(message)
    write_log('\n' + message)
    proc = subprocess.run(
        cmd,
        cwd=str(cwd or PROJECT_ROOT),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        write_log(proc.stdout)
    if check and proc.returncode != 0:
        print_log_tail()
        raise RuntimeError(
            f'Comando fallo con exit={proc.returncode}: {message}. '
            f'Log completo: {SETUP_LOG}'
        )
    return proc


def run_shell(script, *, cwd=None, env=None, check=True):
    return run(['bash', '-lc', script], cwd=cwd, env=env, check=check)


def venv_python_path():
    if os.name == 'nt':
        return VENV_DIR / 'Scripts' / 'python.exe'
    return VENV_DIR / 'bin' / 'python'


def python_info(python):
    python = str(python)
    if not Path(python).exists() and python != sys.executable:
        return None
    code = """
import json
import sys
print(json.dumps({
    'executable': sys.executable,
    'version': sys.version.split()[0],
    'version_info': list(sys.version_info[:3]),
}))
"""
    result = subprocess.run([python, '-c', code], capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout)


def same_executable(left, right):
    try:
        return Path(left).resolve() == Path(right).resolve()
    except Exception:
        return str(left) == str(right)


def setup_env():
    env = os.environ.copy()
    home_bin = str(Path.home() / '.local' / 'bin')
    env['PATH'] = home_bin + os.pathsep + env.get('PATH', '')
    return env


def ensure_uv(env):
    uv = shutil.which('uv', path=env.get('PATH'))
    if uv:
        return uv
    run_shell('curl -LsSf https://astral.sh/uv/install.sh | sh', env=env)
    uv = shutil.which('uv', path=env.get('PATH'))
    if uv:
        return uv
    candidate = Path.home() / '.local' / 'bin' / 'uv'
    if candidate.exists():
        return str(candidate)
    raise RuntimeError('uv no quedo disponible en PATH despues de instalarlo.')


def ensure_project_python39():
    current_info = python_info(sys.executable)
    if current_info and tuple(current_info['version_info'][:2]) == PYTHON_REQUIRED:
        return sys.executable

    project_python = venv_python_path()
    project_info = python_info(project_python)
    if project_info and tuple(project_info['version_info'][:2]) == PYTHON_REQUIRED:
        return str(project_python)

    if platform.system() == 'Windows':
        raise RuntimeError(
            'El kernel actual no es Python 3.9. En Windows selecciona '
            '.venv39-citylearn-v3 como kernel o recrea el entorno segun docs/architecture/local.md.'
        )

    env = setup_env()
    uv = ensure_uv(env)
    print(
        f'Kernel actual: Python {sys.version.split()[0]} ({sys.executable}). '
        f'Creando entorno de proyecto Python 3.9 en {VENV_DIR}.'
    )
    run([uv, 'python', 'install', '3.9'], cwd=PROJECT_ROOT, env=env)
    run([uv, 'venv', '--python', '3.9', str(VENV_DIR)], cwd=PROJECT_ROOT, env=env)

    project_info = python_info(project_python)
    if not project_info or tuple(project_info['version_info'][:2]) != PYTHON_REQUIRED:
        raise RuntimeError(f'No se pudo crear un Python 3.9 valido en {project_python}')
    return str(project_python)


def pip_install(*args):
    cmd = [PROJECT_PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    run(cmd)


def installed_version(package):
    code = """
import importlib.metadata as im
import sys
package = sys.argv[1]
names = (package, package.replace('-', '_'), package.replace('_', '-'))
for name in dict.fromkeys(names):
    try:
        print(im.version(name))
        raise SystemExit(0)
    except im.PackageNotFoundError:
        pass
raise SystemExit(1)
"""
    result = subprocess.run([PROJECT_PYTHON, '-c', code, package], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None


def torch_gpu_ready():
    """True solo si PyTorch puede ejecutar un kernel real en la GPU (detecta sm_120 sin cu128)."""
    code = """
import json, sys
try:
    import torch
    info = {
        'version': torch.__version__,
        'cuda_runtime': getattr(torch.version, 'cuda', None),
        'cuda_available': bool(torch.cuda.is_available()),
    }
    if not info['cuda_available']:
        print(json.dumps({**info, 'ready': False, 'reason': 'cuda_unavailable'}))
        raise SystemExit(0)
    cap = torch.cuda.get_device_capability(0)
    info['device'] = torch.cuda.get_device_name(0)
    info['capability'] = list(cap)
    upper = info['device'].upper()
    needs_cu128 = cap[0] >= 12 or any(k in upper for k in ('BLACKWELL', 'RTX PRO 6000', 'RTX 50'))
    info['needs_cu128'] = needs_cu128
    cuda_rt = str(info['cuda_runtime'] or '')
    if needs_cu128 and not cuda_rt.startswith('12.8'):
        print(json.dumps({**info, 'ready': False, 'reason': 'blackwell_needs_cu128'}))
        raise SystemExit(0)
    x = torch.zeros(1, device='cuda')
    _ = (x + 1).item()
    torch.cuda.synchronize()
    print(json.dumps({**info, 'ready': True, 'kernel_ok': True}))
except Exception as exc:
    print(json.dumps({'ready': False, 'reason': 'kernel_failed', 'error': repr(exc)}))
"""
    result = subprocess.run([PROJECT_PYTHON, '-c', code], capture_output=True, text=True)
    line = (result.stdout or '').strip().splitlines()
    if not line:
        return False, {}
    try:
        data = json.loads(line[-1])
    except Exception:
        return False, {}
    if result.stdout.strip():
        print('[torch]', line[-1])
    return bool(data.get('ready')), data


def torch_cuda_available():
    ready, _ = torch_gpu_ready()
    return ready


def restart_runtime(reason):
    print(f'\n[RESTART REQUERIDO] {reason}')
    print('Reinicia el runtime y vuelve a ejecutar desde la celda 1.2b.')
    try:
        from IPython.core.getipython import get_ipython
        import time

        print('Colab detectado: reiniciando kernel automaticamente...')
        _ip = get_ipython()
        _kernel = getattr(_ip, 'kernel', None) if _ip is not None else None
        if _kernel is not None:
            _kernel.do_shutdown(restart=True)
        time.sleep(10)
    except Exception:
        pass
    raise RuntimeError(reason)


def repair_binary_abi():
    print('[ABI] Reinstalando ruedas binarias con versiones fijadas (numpy 1.23.5)...')
    pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS)


def verify_subprocess_imports():
    result = subprocess.run([PROJECT_PYTHON, '-c', ABI_CHECK], capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout, end='')
    if result.returncode != 0:
        stderr = result.stderr.strip()
        abi_mismatch = any(
            token in stderr
            for token in ('numpy.dtype size changed', 'binary incompatibility', 'numpy.core', 'numpy.strings')
        )
        if abi_mismatch:
            repair_binary_abi()
            result = subprocess.run([PROJECT_PYTHON, '-c', ABI_CHECK], capture_output=True, text=True)
            if result.stdout.strip():
                print(result.stdout, end='')
        if result.returncode != 0:
            if result.stderr.strip():
                print('[STDERR verificacion ABI:]')
                print(result.stderr[-4000:], end='')
            print_log_tail()
            raise RuntimeError('ABI fallo: ' + result.stderr.strip()[-2000:])
    elif result.stderr.strip():
        print('[advertencias ABI (no criticas):]')
        print(result.stderr.strip())


def verify_current_kernel_imports_if_needed():
    if not same_executable(PROJECT_PYTHON, sys.executable):
        print(
            f'Kernel notebook: Python {sys.version.split()[0]} ({sys.executable}). '
            f'Entrenamiento: {PROJECT_PYTHON}. No se importan paquetes del proyecto en el kernel.'
        )
        return

    modules = ('numpy', 'scipy', 'sklearn', 'pandas', 'citylearn.v3.environment')
    failures = {}
    for module_name in modules:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[module_name] = repr(exc)
    if failures:
        restart_runtime(
            'El kernel actual tiene imports binarios inconsistentes: '
            f'{failures}. Esto ocurre si pip cambio numpy/scipy sin reiniciar.'
        )


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_DIR no existe: {PROJECT_ROOT}. Ejecuta primero la celda 1.2.')

SETUP_LOG.write_text('', encoding='utf-8')
PROJECT_PYTHON = ensure_project_python39()
PYTHON = PROJECT_PYTHON
project_info = python_info(PROJECT_PYTHON)
if not project_info or tuple(project_info['version_info'][:2]) != PYTHON_REQUIRED:
    raise RuntimeError(f'Python de proyecto invalido: {project_info}')

os.chdir(PROJECT_DIR)
CONSTRAINTS.write_text('\n'.join(f'{p}=={v}' for p, v in PINNED.items()) + '\n')
print(f"Python proyecto: {project_info['version']} ({PROJECT_PYTHON})")
print(f"Python kernel  : {sys.version.split()[0]} ({sys.executable})")
print(f'Log setup      : {SETUP_LOG}')

# Pip compatible con gym/ray legacy del proyecto.
run([PROJECT_PYTHON, '-m', 'ensurepip', '--upgrade'], check=False)
pip_install('--force-reinstall', 'pip==21.3.1', 'setuptools==65.5.0', 'wheel==0.38.0')

pip_install('-q', *BASE_DEPS)
for package in NO_DEPS_UTILS:
    if installed_version(package.split('==')[0]) is None:
        pip_install('-q', '--no-deps', package)

ready, torch_info = torch_gpu_ready() if installed_version('torch') else (False, {})
if not ready:
    reason = torch_info.get('reason', 'missing_or_incompatible')
    print(f'[torch] Instalando PyTorch ({PYTORCH_INDEX_URL.split("/")[-1]}) — motivo: {reason}')
    pip_install('--force-reinstall', '-q', *TORCH_PACKAGES, '--index-url', PYTORCH_INDEX_URL)
    ready, torch_info = torch_gpu_ready()
if not ready:
    raise RuntimeError(
        'PyTorch no puede ejecutar kernels en esta GPU. '
        f'info={torch_info}. Blackwell (RTX PRO 6000) requiere cu128: '
        'pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128'
    )

for package_dir in EDITABLES:
    pip_install('-q', '--no-deps', '-c', str(CONSTRAINTS), '-e', package_dir)

# Reinstalacion final: evita ruedas pandas/scipy compiladas para numpy 2.x.
pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS)

binary_after = {package: installed_version(package) for package in BINARY_DEPS}
print('Paquetes binarios:', binary_after)

print('\nVerificando ABI en Python 3.9 del proyecto...')
verify_subprocess_imports()
verify_current_kernel_imports_if_needed()
print('\nCelda 1.3 OK: Python 3.9 del proyecto listo y backends en modo editable.')

In [ ]:
# 1.4 Configurar sys.path, CUDA y smoke imports
# Vinculada con 1.3: usa PROJECT_PYTHON aunque el kernel Colab sea Python 3.11.
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_DIR', '/content/MADRLCitytleranflexresdr'))
if not PROJECT_ROOT.exists() and Path.cwd().name == 'MADRLCitytleranflexresdr':
    PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.resolve()
REPO = str(PROJECT_ROOT)

PYTHON_MIN = globals().get('PYTHON_MIN', (3, 9))
PYTHON_MAX_EXCLUSIVE = globals().get('PYTHON_MAX_EXCLUSIVE', (3, 10))
PROJECT_PYTHON = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
PYTHON = PROJECT_PYTHON
EDITABLES = globals().get('EDITABLES', [
    'CityLearn/',
    'external/HARL/',
    'external/off-policy/',
    'external/MAAC/',
    'external/MARL/src/',
])


def repo_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def _python_info_strict(python):
    code = """
import json
import sys
print(json.dumps({
    'executable': sys.executable,
    'version': sys.version.split()[0],
    'version_info': list(sys.version_info[:3]),
}))
"""
    result = subprocess.run([str(python), '-c', code], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr or f'No se pudo ejecutar {python}')
    return json.loads(result.stdout)


python_info = _python_info_strict


if 'same_executable' not in globals():
    def same_executable(left, right):
        try:
            return Path(left).resolve() == Path(right).resolve()
        except Exception:
            return str(left) == str(right)

if 'restart_runtime' not in globals():
    def restart_runtime(reason):
        print(f'\n[RESTART REQUERIDO] {reason}')
        print('Reinicia el runtime y vuelve a ejecutar desde la celda 1.2b.')
        try:
            from IPython.core.getipython import get_ipython
            import time

            print('Colab detectado: reiniciando kernel automaticamente...')
            _ip = get_ipython()
            _kernel = getattr(_ip, 'kernel', None) if _ip is not None else None
            if _kernel is not None:
                _kernel.do_shutdown(restart=True)
            time.sleep(10)
        except Exception:
            pass
        raise RuntimeError(reason)


project_python_info = python_info(PROJECT_PYTHON)
if project_python_info is None:
    raise RuntimeError(f'No se pudo leer Python de proyecto: {PROJECT_PYTHON}')
if not (PYTHON_MIN <= tuple(project_python_info['version_info'][:2]) < PYTHON_MAX_EXCLUSIVE):
    raise RuntimeError(
        f"Python de proyecto {project_python_info['version']} no soportado. "
        'Ejecuta primero la celda 1.3 para crear/validar .venv39-citylearn-v3.'
    )

PATHS = list(dict.fromkeys(str(path) for path in [
    PROJECT_ROOT,
    PROJECT_ROOT / 'CityLearn',
    PROJECT_ROOT / 'CityLearn' / 'scripts',
    *(repo_path(path) for path in EDITABLES),
]))
missing = [path for path in PATHS if not Path(path).exists()]
if missing:
    raise FileNotFoundError(f'Rutas requeridas no encontradas: {missing}')

for path in reversed(PATHS):
    while path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

old_pythonpath = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
old_pythonpath = [p for p in old_pythonpath if p not in PATHS]
os.environ['PYTHONPATH'] = os.pathsep.join(PATHS + old_pythonpath)
os.environ['CITYLEARN_PROJECT_ROOT'] = REPO
os.environ.setdefault('CUDA_DEVICE_ORDER', 'PCI_BUS_ID')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('PYTHONHASHSEED', '0')

SMOKE_IMPORTS = {
    'torch': 'torch',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'sklearn',
    'citylearn': 'citylearn',
    'citylearn.v3.environment': 'citylearn.v3.environment',
    'harl': 'harl',
    'runner_msac': 'runner_msac',
    'offpolicy': 'offpolicy',
    'algorithms.attention_sac': 'algorithms.attention_sac',
}
OPTIONAL_IMPORTS = {'harl', 'runner_msac', 'offpolicy', 'algorithms.attention_sac'}

smoke_code = f"""
import importlib, json, sys
paths = {PATHS!r}
modules = {SMOKE_IMPORTS!r}
optional = set({sorted(OPTIONAL_IMPORTS)!r})
for path in reversed(paths):
    if path not in sys.path:
        sys.path.insert(0, path)
imports, versions = {{}}, {{'python': sys.version.split()[0], 'executable': sys.executable}}
for label, module_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        imports[label] = 'ok'
        version = getattr(module, '__version__', None)
        if version:
            versions[label] = version
    except Exception as exc:
        imports[label] = f'FAILED: {{exc}}'
print(json.dumps({{'imports': imports, 'versions': versions}}, indent=2, sort_keys=True))
failed = {{k: v for k, v in imports.items() if v.startswith('FAILED')}}
critical_failed = {{k: v for k, v in failed.items() if k not in optional}}
if failed.keys() - critical_failed.keys():
    print(f'[WARN] Modulos opcionales no disponibles: {{sorted(failed.keys() - critical_failed.keys())}}')
if critical_failed:
    abi = any('numpy.dtype size changed' in v or 'numpy.core' in v or 'numpy.strings' in v or '_center' in v for v in critical_failed.values())
    hint = 'Reinicia el runtime y ejecuta 1.1-1.4 en orden.' if abi else 'Ejecuta primero la celda 1.3 y repite 1.4.'
    raise SystemExit(f'ERROR: imports criticos fallaron: {{critical_failed}}. {{hint}}')
"""

result = subprocess.run([PROJECT_PYTHON, '-c', smoke_code], capture_output=True, text=True, env=os.environ.copy())
if result.stdout.strip():
    print(result.stdout, end='')
if result.returncode != 0:
    if result.stderr.strip():
        print('[STDERR smoke check:]')
        print(result.stderr, end='')
    raise RuntimeError('Smoke imports criticos fallaron en Python 3.9 del proyecto. Revisa el JSON anterior.')

if same_executable(PROJECT_PYTHON, sys.executable):
    current_failures = {}
    for label, module_name in SMOKE_IMPORTS.items():
        if label in OPTIONAL_IMPORTS:
            continue
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            current_failures[label] = repr(exc)
    if current_failures:
        restart_runtime(
            'El subprocess importa bien, pero el kernel actual esta inconsistente: '
            f'{current_failures}.'
        )
else:
    print(f'Kernel notebook en {sys.version.split()[0]}; smoke imports ejecutados con {PROJECT_PYTHON}.')

print(f'Celda 1.4 OK: sys.path, CUDA env y smoke imports configurados para {PROJECT_PYTHON}.')


### Persistencia en Google Drive

La campaña escribe en `outputs/campaigns/iquitos_v1/` dentro del runtime. Después de ejecutar una etapa, la celda **7.2** copia esa raíz a `MyDrive/MADRL_campaigns/iquitos_v1/`. Drive es solo una copia: no se reanuda desde Drive ni se lee ningún resultado anterior a esta campaña.

In [ ]:
# ── 1.5  Montar Google Drive (solo Colab; destino de la copia de intentos) ──
from pathlib import Path

DRIVE_ROOT = None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/MADRL_campaigns/iquitos_v1')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print('Copia de intentos en Drive:', DRIVE_ROOT)
else:
    print('Entorno local: sin Drive; los intentos quedan en outputs/campaigns/iquitos_v1/.')

## Sección 2: Configuración de la campaña

### 2.1 Campaña `fresh_iquitos_20261101_v1`

La celda **2.1** lee `configs/experiments/iquitos.yaml` y fija las rutas de la campaña. No busca la carpeta más nueva ni sigue punteros: la raíz de artefactos sale del alias de almacenamiento (`outputs/campaigns/iquitos_v1/`).

| Clave | Significado |
|---|---|
| `campaign_id` / `storage_alias` | identidad de la campaña y nombre corto de sus carpetas |
| `dataset_root` | dataset congelado `data/campaigns/iquitos_v1` |
| `seeds.development` / `seeds.confirmatory` | 901-903 para HPO; 1001-1012 para confirmación |
| `stages` | episodios, pasos, semillas y escenarios por etapa |
| `resource_authorization.authorized_stages` | etapas que `--execute` puede entrenar (vacía por defecto) |

In [ ]:
# ── 2.1  Configuracion y rutas de la campaña nueva ─────────────────────────────
import os
import subprocess
import sys
from pathlib import Path

import yaml

# Importa JSON para leer la salida de la CLI de campaña.
import json

# Detecta la raiz del proyecto padre: contiene uc3m/ y configs/experiments/.
def find_project_root(start=None):
    # Parte del directorio actual del notebook.
    start = Path(start or Path.cwd()).resolve()
    # Recorre el directorio actual, sus padres y la ruta estandar de Colab.
    for candidate in (start, *start.parents, Path('/content/MADRLCitytleranflexresdr')):
        # Acepta la primera carpeta con la estructura del proyecto.
        if (candidate / 'uc3m').is_dir() and (candidate / 'configs' / 'experiments').is_dir():
            return candidate
    # Falla de forma visible si el notebook no esta dentro del proyecto.
    raise FileNotFoundError('No se encontro la raiz del proyecto (uc3m/ + configs/experiments/).')

# Fija la raiz del proyecto padre.
PROJECT_ROOT = find_project_root()

# Fija la raiz del simulador CityLearn (dependencia protegida, solo lectura).
CITYLEARN_ROOT = PROJECT_ROOT / 'CityLearn'

# Agrega proyecto y CityLearn al path de Python del kernel.
for _path in (str(CITYLEARN_ROOT), str(PROJECT_ROOT)):
    if _path not in sys.path:
        sys.path.insert(0, _path)

# Usa el Python 3.9 del proyecto si una celda previa lo definio.
PYTHON = globals().get('PROJECT_PYTHON', sys.executable)

# Configuracion unica de la campaña nueva.
CAMPAIGN_CONFIG = 'configs/experiments/iquitos.yaml'

# Dataset congelado de la campaña (splits train_2023 / validation_2024 / test_2025).
DATASET_ROOT = PROJECT_ROOT / 'data/campaigns/iquitos_v1'

# Raiz de artefactos de la campaña nueva; no existe hasta ejecutar --stage prepare.
ARTIFACT_ROOT = PROJECT_ROOT / 'outputs/campaigns/iquitos_v1'

# Manifiesto de la campaña escrito por la etapa prepare.
MANIFEST_PATH = ARTIFACT_ROOT / 'campaign_manifest.json'

# Entorno de los subprocesos: la CLI importa uc3m y CityLearn desde el arbol local.
def cli_env():
    env = dict(os.environ)
    env['PYTHONPATH'] = os.pathsep.join([str(PROJECT_ROOT), str(CITYLEARN_ROOT), env.get('PYTHONPATH', '')])
    return env

# Muestra el codigo de salida y el JSON que imprime la CLI de campaña.
def show_cli_result(result):
    print('exit code =', result.returncode)
    lines = [line for line in (result.stdout or '').splitlines() if line.strip()]
    if lines:
        try:
            print(json.dumps(json.loads(lines[-1]), indent=2))
        except ValueError:
            print('\n'.join(lines[-20:]))
    if result.returncode != 0 and result.stderr:
        print(result.stderr[-3000:])

# Muestra las rutas clave para diagnostico.
print('PROJECT_ROOT  =', PROJECT_ROOT)
print('CAMPAIGN      =', PROJECT_ROOT / CAMPAIGN_CONFIG)
print('DATASET_ROOT  =', DATASET_ROOT, '(existe)' if DATASET_ROOT.is_dir() else '(FALTA)')
print('ARTIFACT_ROOT =', ARTIFACT_ROOT, '(preparada)' if MANIFEST_PATH.is_file() else '(aun no preparada)')

# Lee la configuracion de la campaña.
CONFIG = yaml.safe_load((PROJECT_ROOT / CAMPAIGN_CONFIG).read_text(encoding='utf-8'))
print('\ncampaign_id      =', CONFIG['campaign_id'])
print('storage_alias    =', CONFIG['storage_alias'])
print('algoritmos       =', CONFIG['algorithms'])
print('escenarios       =', CONFIG['scenarios'])
print('semillas         =', CONFIG['seeds'])
print('splits           =', CONFIG['splits'])
for stage, spec in CONFIG['stages'].items():
    print(f'etapa {stage:12s}', spec)
print('etapas autorizadas para ejecutar:', CONFIG['resource_authorization'].get('authorized_stages') or 'ninguna')

### 2.1b Estado de la campaña

Lista las etapas del manifiesto y, si una etapa se ejecutó, su `stages/<etapa>_result.json`. Si la campaña aún no está preparada, lo dice; no hay estados heredados de corridas anteriores.

In [ ]:
# Lee el estado real de la campaña desde sus manifiestos; nunca desde corridas anteriores.
if not MANIFEST_PATH.is_file():
    # Sin manifiesto no hay campaña preparada ni entrenamiento que reportar.
    print('Campaña no preparada: ejecuta la etapa prepare. No hay resultados de entrenamiento.')
else:
    # Carga el manifiesto de la campaña.
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    # Muestra identidad y estado declarados.
    print('campaign_id =', manifest['campaign_id'], '| status =', manifest['status'])
    # Recorre las etapas declaradas en el manifiesto.
    for stage, record in manifest['stages'].items():
        # Cada etapa ejecutada escribe stages/<stage>_result.json.
        result_path = ARTIFACT_ROOT / 'stages' / f'{stage}_result.json'
        if not result_path.is_file():
            print(f"{stage:12s} {record['treatments']:4d} tratamientos planificados | sin ejecutar")
            continue
        result = json.loads(result_path.read_text(encoding='utf-8'))
        print(f"{stage:12s} {result['completed_treatments']}/{result['planned_treatments']} completos | "
              f"intentos fallidos={result['failed_attempts']} | status={result['status']}")

## Sección 3: Dataset de la campaña `iquitos_v1`

**17 edificios** · dataset construido desde fuentes primarias auditadas y dividido por año calendario. Ningún dataset horario anterior entra a la campaña.

| Recurso | Detalle |
|---|---|
| Construcción | `tools/dataset/build_fresh_campaign_dataset.py` → `dataset_acceptance.json` (`dataset_verified`, `dataset_hash`) |
| Splits | `tools/dataset/split_campaign.py` → `train_2023` (8 760 h), `validation_2024` (8 784 h), `test_2025` (8 760 h) |
| Normalizador | `normalizer.json`, límites ajustados solo con 2023 |
| Uso | entrenamiento en 2023 · selección de hiperparámetros en 2024 · evaluación congelada en 2025 |
| Equipos | capacidades fijadas por diseño en `split_manifest.json` (`fixed_device_sizes`) |

In [ ]:
# ── 3.1  Verificar el dataset congelado de la campaña ──────────────────────────
# Solo lectura: no modifica ningun archivo del dataset.
import json
from pathlib import Path

DATASET_ROOT = Path(globals().get('DATASET_ROOT', Path(globals().get('PROJECT_DIR', Path.cwd())) / 'data/campaigns/iquitos_v1'))
acceptance = json.loads((DATASET_ROOT / 'dataset_acceptance.json').read_text(encoding='utf-8'))
split_manifest = json.loads((DATASET_ROOT / 'split_manifest.json').read_text(encoding='utf-8'))

print('campaign_id      :', acceptance['campaign_id'])
print('dataset_verified :', acceptance['dataset_verified'])
print('dataset_hash     :', acceptance['dataset_hash'][:16], '...')
print('balance checks   :', acceptance['balance_checks'], '| fallas:', acceptance['balance_failure_count'])
print('limitaciones     :', len(acceptance.get('limitations', [])))
if not acceptance['dataset_verified']:
    raise RuntimeError('El dataset de la campaña no esta verificado; no se puede preparar la campaña.')
if split_manifest['dataset_hash'] != acceptance['dataset_hash']:
    raise RuntimeError('split_manifest.json no corresponde al dataset aceptado.')

print(f"\n{'Split':<16} {'Inicio':<26} {'Filas':>6} {'CSV':>5}  schema")
for split in split_manifest['splits']:
    folder = DATASET_ROOT / 'splits' / split['directory']
    schema_ok = (DATASET_ROOT / 'splits' / split['schema_path']).is_file()
    print(f"{split['split_id']:<16} {split['start']:<26} {split['rows']:>6} {len(list(folder.glob('*.csv'))):>5}  "
          f"{'OK' if schema_ok else 'FALTA'}")
print('\nedificios con equipos fijados:', len(split_manifest.get('fixed_device_sizes', {})))

## Sección 4: Entorno Dec-POMDP — 17 agentes

**Dec-POMDP:** cada edificio es un agente con observación parcial local.
**CTDE:** el crítico usa el estado global durante entrenamiento; la ejecución es completamente local.

```
Dataset: data/campaigns/iquitos_v1/splits/train_2023 (17 edificios, 2023)

Observación local oᵢ(t): HETEROGÉNEA por edificio (varía según nº de chargers EV)
  └── las dimensiones exactas por edificio las imprime la celda 4.1

  Componentes por agente:
  ├── Tiempo (mes, hora, day_type)
  ├── Física edificio (non_shiftable_load, dhw_demand, cooling_demand, solar_generation)
  ├── BESS (SOC, nominal_power, acciones previas)
  ├── EV por charger (SOC_k, departure_time_k, required_soc_k, estimated_arrival_k, state_k)
  └── Señales globales (carbon_intensity, electricity_pricing, outdoor_dry_bulb_temp,
                        diffuse_solar_irradiance, direct_solar_irradiance)

Acción local aᵢ(t): HETEROGÉNEA por edificio
  └── BESS + potencia por cargador EV + cargas flexibles (celda 4.1)

Nota HAPPO: share_param=False — cada edificio tiene política INDEPENDIENTE
            (patrones locales distintos: industrial vs residencial).
```

> **MAPPO no es baseline ni algoritmo de este flujo.** Solo antecedente metodológico de políticas compartidas; los 4 MADRL oficiales son HAPPO, MASAC, MATD3 y MAAC.

In [ ]:
# ── 4.1  Smoke test del entorno Dec-POMDP con el split train_2023 ──────────────
# Se pasa el schema del split de entrenamiento explicitamente; nunca el schema por defecto del paquete.
import json
import os
import subprocess
import sys

PYTHON = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
REPO = globals().get('REPO', str(globals().get('PROJECT_ROOT', '/content/MADRLCitytleranflexresdr')))

smoke_code = r'''
import json, sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "CityLearn"))
from citylearn.v3.environment import make_citylearn_v3_env, describe_environment
from citylearn.v3.config import CityLearnV3ExperimentConfig
SCHEMA = os.path.join(os.getcwd(), "data/campaigns/iquitos_v1/splits/train_2023/schema.json")
assert os.path.exists(SCHEMA), f"Schema de la campaña no encontrado: {SCHEMA}"
cfg = CityLearnV3ExperimentConfig(schema_path=SCHEMA)
results = {}
for scenario in ("E1", "E2", "E3"):
    env = make_citylearn_v3_env(cfg, schema_path=SCHEMA, scenario=scenario, seed=0, episode_time_steps=4,
                                reward_aggregation="team_mean", normalize_observations=True,
                                madrl_algorithm="MATD3", use_citylearn_v3_reward=True)
    try:
        desc = describe_environment(env)
        obs, info = env.reset()
        agents = list(obs.keys())
        acts = {a: env.action_space(a).sample() for a in env.agents}
        obs2, rews, terms, truncs, infos = env.step(acts)
        results[scenario] = {
            "num_agents": desc["num_agents"],
            "obs_dim_agent_0": len(obs[agents[0]]),
            "obs_dims_range": [min(len(v) for v in obs.values()), max(len(v) for v in obs.values())],
            "reward_function": desc.get("reward_function", "N/A"),
            "reward_mean_step1": round(sum(float(r) for r in rews.values()) / len(rews), 5),
        }
    finally:
        env.close()
print(json.dumps(results, indent=2, default=str))
'''

result = subprocess.run([PYTHON, '-c', smoke_code], cwd=REPO, capture_output=True, text=True, env=os.environ.copy())
stderr_lines = [line for line in (result.stderr or '').splitlines() if not line.startswith('INFO:')]
if stderr_lines:
    print('\n'.join(stderr_lines[-20:]))
if result.returncode != 0:
    raise RuntimeError(f'Smoke-test CityLearn v3 fallo (exit={result.returncode})\n{result.stderr[-800:]}')

results = json.loads(result.stdout[result.stdout.index('{'):])
print(f"{'Escenario':<10} {'Agentes':>8} {'Obs(ag0)':>9} {'Obs min-max':>12} {'Rew(s1)':>10}")
print('-' * 54)
for sc, r in results.items():
    lo, hi = r['obs_dims_range']
    print(f"{sc:<10} {r['num_agents']:>8} {r['obs_dim_agent_0']:>9} {f'{lo}-{hi}':>12} {r['reward_mean_step1']:>10}")
print('\nSmoke test: 1 paso aleatorio por escenario; no es entrenamiento.')

## Sección 5: Función de recompensa multiobjetivo

La recompensa **durante entrenamiento** pondera flexibilidad/CO₂/costo por escenario (E1/E2/E3) con el perfil `fresh_kpi_aligned_v5` de `CityLearnV3MADRLRewardFunction`. Los pesos salen del catálogo único `configs/scenarios.yaml` → `configs/objective_policies.yaml`; la evaluación post-entrenamiento (secciones 8-9) usa el catálogo de KPIs `configs/evaluation/kpis.yaml`.

### Componentes de recompensa

| Componente | Descripción |
|---|---|
| **Flexibilidad** | pico convexo + rampa lineal de la potencia neta coincidente del distrito |
| **CO₂** | importación del edificio × intensidad de carbono |
| **Costo** | importación del edificio × precio, sin crédito de exportación |
| **EV** | penalización de servicio/violación, fuera del vector ponderado |
| **BESS** | penalización de ciclado, fuera del vector ponderado |

### Pesos por escenario (entrenamiento)

| Escenario | flex | carbon | cost |
|:---:|:---:|:---:|:---:|
| **E1** | **0.60** | 0.15 | 0.25 |
| **E2** | 0.25 | **0.60** | 0.15 |
| **E3** | 0.15 | 0.25 | **0.60** |

### Recompensa de equipo (CTDE)
El envoltorio Dec-POMDP aplica `reward_aggregation = "team_mean"`: los 17 agentes reciben el mismo pago común. Cada intento escribe `supervision.json` y se rechaza si los términos EV/BESS dominan los ejes ponderados (`supervision.max_constraint_dominance_ratio` en `configs/experiments/iquitos.yaml`). La etapa `sensitivity` cambia la dominancia del eje principal a p = 0.55 / 0.65.

In [ ]:
# ── 5.1  Visualizar pesos de recompensa por escenario ────────────────────
# Fuente: configs/objective_policies.yaml (catalogo unico de pesos E1/E2/E3).
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

_root = Path(globals().get('PROJECT_ROOT', globals().get('PROJECT_DIR', Path.cwd())))
policies = yaml.safe_load((_root / 'configs/objective_policies.yaml').read_text(encoding='utf-8'))
AXES = policies['axis_order']
WEIGHTS = {scenario: [policy['weights'][axis] for axis in AXES] for scenario, policy in policies['policies'].items()}
COLORS = ["#3b82f6", "#22c55e", "#f59e0b"]

fig, axes = plt.subplots(1, len(WEIGHTS), figsize=(13, 5.0), sharey=True)
fig.suptitle("Pesos de recompensa por escenario (configs/objective_policies.yaml)", fontsize=13, fontweight="bold")
for ax, (scenario, values) in zip(axes, WEIGHTS.items()):
    bars = ax.bar(AXES, values, color=COLORS, edgecolor="white", linewidth=1.5, width=0.55)
    ax.set_title(f"Escenario {scenario}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 0.85)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{value:.2f}", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

for scenario, values in WEIGHTS.items():
    print(f"{scenario}: " + " + ".join(f"{w:.2f}·{axis}" for axis, w in zip(AXES, values)) + "   (agregacion team_mean)")

## Sección 6: Hiperparámetros y protocolo de selección

### Selección en validación, confirmación con configuración congelada

| Paso | Etapa | Semillas | Episodios x pasos | Split de decisión |
|:---:|:---|:---:|:---:|:---|
| 1 | `pilot` | 901 | 2 x 168 | mide recursos; no selecciona |
| 2 | `hpo` | 901-903 | 10 x 8 760 | `validation_2024` |
| 3 | congelado | — | — | `configuration_freezes.json` por algoritmo |
| 4 | `confirm` | 1001-1012 | 50 x 8 760 | evaluación congelada en `test_2025` |
| 5 | `sensitivity` | 1001-1003 | 50 x 8 760 | dominancia p = 0.55 / 0.65 |

`confirm` y `sensitivity` se niegan a correr sin el congelado de configuración. Ninguna etapa reanuda intentos.

### 6.1 Fuente única de hiperparámetros (`configs/algorithms/native.yaml`)

La rejilla común varía `learning_rate` (0.0001, 0.0003), `hidden_dimension` (128, 256) y `gamma` (0.99, 0.9999): 8 combinaciones por algoritmo. Cada clave se traduce a banderas que el backend nativo realmente interpreta; una clave desconocida falla en lugar de ignorarse.

| Algoritmo | Nota declarada en la configuración |
|---|---|
| **HAPPO** | rejilla completa |
| **MASAC** | `rnn_hidden_dim` limitado a 64 con episodios de 8 760 pasos: 4 configuraciones efectivas, no 8 |
| **MATD3** | buffer total repartido entre las 17 políticas |
| **MAAC** | rejilla completa |

Presupuesto según `docs/architecture/runbook.md`: HPO 252 tratamientos, confirmación 144 y sensibilidad 72. La celda **6.1** lo lee del manifiesto cuando la campaña está preparada.

In [ ]:
# ── 6.1  Rejilla de hiperparametros y presupuesto de la campaña ───────────────
import json
from pathlib import Path

import yaml

_root = Path(globals().get('PROJECT_ROOT', globals().get('PROJECT_DIR', Path.cwd())))
native = yaml.safe_load((_root / 'configs/algorithms/native.yaml').read_text(encoding='utf-8'))
print('runtime_flags       :', native['runtime_flags'])
print('hyperparameter_grid :', native['hyperparameter_grid'])
for algorithm, spec in native['algorithms'].items():
    print(f"\n{algorithm}")
    print('  mapping     :', spec['mapping'])
    print('  fijos       :', spec.get('fixed') or {})
    if spec.get('native_clamps'):
        print('  clamps      :', spec['native_clamps'])
    for limitation in spec.get('declared_limitations') or []:
        print('  limitacion  :', limitation)

_manifest = _root / 'outputs/campaigns/iquitos_v1/campaign_manifest.json'
if _manifest.is_file():
    manifest = json.loads(_manifest.read_text(encoding='utf-8'))
    print('\nPresupuesto (manifiesto):', json.dumps(manifest['budget'], indent=2))
    print('Candidatos HPO efectivos por algoritmo:', {k: len(v) for k, v in manifest['hpo_candidates_by_algorithm'].items()})
else:
    print('\nCampaña no preparada: el presupuesto aparece tras la celda 7.1 (etapa prepare).')

### 6.2 Plan piloto — 4 algoritmos, E1, semilla 901

> **El piloto verifica el pipeline y mide recursos.** No es un resultado de tesis.

La celda consulta la etapa `pilot` con la CLI **sin** `--execute`: muestra el plan y no entrena. Ejecutar el piloto se hace en **7.2** con `EXECUTE_STAGE = True`, solo si la etapa está autorizada.

In [ ]:
# ── 6.2  Plan piloto (CLI sin --execute: no entrena) ──────────────────────────
pilot_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                 '--config', 'configs/experiments/iquitos.yaml', '--stage', 'pilot']
print(' '.join(map(str, pilot_command)))
pilot_result = subprocess.run(pilot_command, cwd=PROJECT_ROOT, env=cli_env(),
                              capture_output=True, text=True, check=False)
show_cli_result(pilot_result)

## Sección 7: Lanzamiento por la CLI de campaña

Única vía: `python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage <etapa> [--execute]`. La CLI verifica que las entradas de la campaña preparada no cambiaron, crea cada intento vacío y llama al backend nativo a través de `uc3m.adapters.algorithms.native_entrypoint`. No hay lanzadores por algoritmo, por plataforma ni monitores separados.

In [ ]:
# ── 7.0  Helper de lectura de intentos de una etapa ─────────────────────────────
import json

def stage_attempts(stage):
    """Filas (tratamiento, intento) de stages/<stage>_result.json; lista vacia si la etapa no se ejecuto."""
    result_path = ARTIFACT_ROOT / 'stages' / f'{stage}_result.json'
    if not result_path.is_file():
        return []
    result = json.loads(result_path.read_text(encoding='utf-8'))
    return [(treatment, attempt) for treatment in result['treatments'] for attempt in treatment['attempts']]

print('Helper listo. ARTIFACT_ROOT =', ARTIFACT_ROOT)

### 7.1 Preparar la campaña (no entrena)

`--stage prepare` construye el DAG, valida el dataset y escribe el manifiesto con hashes de configuración, dataset, normalizador y código. No consulta la GPU. Si la campaña ya existe y sus entradas cambiaron, la CLI responde `campaign_inputs_changed`: hay que preparar una identidad nueva, nunca reutilizar el manifiesto.

In [ ]:
# Prepara la campaña: escribe manifiestos y presupuesto. NO entrena ni consulta la GPU.
prepare_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                   '--config', 'configs/experiments/iquitos.yaml', '--stage', 'prepare']

# Muestra el comando exacto.
print(' '.join(map(str, prepare_command)))

# Ejecuta la preparacion; si la campaña ya existe, la CLI solo verifica que sus entradas no cambiaron.
prepare_result = subprocess.run(prepare_command, cwd=PROJECT_ROOT, env=cli_env(),
                                capture_output=True, text=True, check=False)

# Muestra el JSON de la CLI (status prepared_not_executed y presupuesto de pasos).
show_cli_result(prepare_result)

### 7.2 Ejecutar una etapa (opt-in explícito)

`EXECUTE_STAGE = False` por defecto: ejecutar todas las celdas nunca entrena. Para entrenar:

1. El propietario añade la etapa en `resource_authorization.authorized_stages` de `configs/experiments/iquitos.yaml` y se prepara de nuevo (cambia la identidad de la configuración).
2. Se fija `RUN_STAGE` (`pilot`, luego `hpo`, `confirm`, `sensitivity`) y `EXECUTE_STAGE = True`.

Al terminar, si hay Drive montado, se copia `outputs/campaigns/iquitos_v1/` a Drive.

In [ ]:
# Etapa a consultar o ejecutar: 'pilot', 'hpo', 'confirm' o 'sensitivity'.
RUN_STAGE = 'pilot'

# Opt-in explicito. Con False la CLI solo informa el plan de la etapa y no entrena.
# Con True solo corre si la etapa figura en resource_authorization.authorized_stages
# de configs/experiments/iquitos.yaml; cada intento empieza desde cero (sin reanudacion).
EXECUTE_STAGE = False

# Construye el comando de la unica via de lanzamiento.
stage_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                 '--config', 'configs/experiments/iquitos.yaml', '--stage', RUN_STAGE]
stage_command += ['--execute'] if EXECUTE_STAGE else []

# Muestra el comando exacto.
print(' '.join(map(str, stage_command)))

# Ejecuta la CLI: sin --execute imprime prepared_not_executed con el plan de la etapa.
stage_result = subprocess.run(stage_command, cwd=PROJECT_ROOT, env=cli_env(),
                              capture_output=True, text=True, check=False)

# Muestra el JSON de la CLI (not_authorized, training_complete o incomplete).
show_cli_result(stage_result)

# Copia la raiz de artefactos a Drive solo despues de una ejecucion real.
if EXECUTE_STAGE and globals().get('DRIVE_ROOT') and ARTIFACT_ROOT.is_dir():
    import shutil
    shutil.copytree(ARTIFACT_ROOT, DRIVE_ROOT / 'outputs', dirs_exist_ok=True)
    print('Copiado a Drive:', DRIVE_ROOT / 'outputs')

### 7.2b Confirmación y sensibilidad

Son etapas de la misma CLI (`--stage confirm`, `--stage sensitivity`), no otra campaña ni otro orquestador. Confirmación: 4 algoritmos x 3 escenarios x 12 semillas 1001-1012 con la configuración congelada en validación. Sensibilidad: dominancia p = 0.55 / 0.65 sobre las semillas 1001-1003, con catálogos de pesos derivados y hasheados en `stages/sensitivity_catalogs/`. Ambas exigen `configuration_freezes.json`.

### 7.3 Estado de las etapas

Lectura de `campaign_manifest.json` y `stages/<etapa>_result.json`. Se puede ejecutar en cualquier momento (también tras reiniciar el kernel después de 2.1).

In [ ]:
# Lee el estado real de la campaña desde sus manifiestos; nunca desde corridas anteriores.
if not MANIFEST_PATH.is_file():
    # Sin manifiesto no hay campaña preparada ni entrenamiento que reportar.
    print('Campaña no preparada: ejecuta la etapa prepare. No hay resultados de entrenamiento.')
else:
    # Carga el manifiesto de la campaña.
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    # Muestra identidad y estado declarados.
    print('campaign_id =', manifest['campaign_id'], '| status =', manifest['status'])
    # Recorre las etapas declaradas en el manifiesto.
    for stage, record in manifest['stages'].items():
        # Cada etapa ejecutada escribe stages/<stage>_result.json.
        result_path = ARTIFACT_ROOT / 'stages' / f'{stage}_result.json'
        if not result_path.is_file():
            print(f"{stage:12s} {record['treatments']:4d} tratamientos planificados | sin ejecutar")
            continue
        result = json.loads(result_path.read_text(encoding='utf-8'))
        print(f"{stage:12s} {result['completed_treatments']}/{result['planned_treatments']} completos | "
              f"intentos fallidos={result['failed_attempts']} | status={result['status']}")

### 7.4 Auditoría de intentos

Para cada intento de cada etapa ejecutada verifica la presencia de `attempt.json`, `child.log`, `learning_evidence.json`, `supervision.json` y `completion.json`, y muestra las razones de rechazo registradas.

In [ ]:
# ── 7.4  Auditoria de intentos de la campaña nueva ──────────────────────────────
EXPECTED = ('attempt.json', 'child.log', 'learning_evidence.json', 'supervision.json', 'completion.json')
_any = False
for stage in ('pilot', 'hpo', 'confirm', 'sensitivity'):
    for treatment, attempt in stage_attempts(stage):
        _any = True
        folder = ARTIFACT_ROOT / attempt['path']
        present = [name for name in EXPECTED if (folder / name).is_file()]
        status = 'COMPLETO' if attempt.get('training_complete') else 'RECHAZADO'
        print(f"{status:9s} {treatment['treatment_id']} [{attempt['attempt_id']}] "
              f"archivos {len(present)}/{len(EXPECTED)} razones={attempt.get('reasons', [])}")
if not _any:
    print('Ninguna etapa ejecutada en esta campaña: no hay intentos que auditar.')

### 7.5 Revisar fallos

| Motivo | Significado |
|---|---|
| `no_successful_update:<owner>` | El optimizador no aplicó ninguna actualización con gradientes finitos |
| `parameters_unchanged:<owner>` | Hubo pasos, pero los parámetros no cambiaron |
| `completed_with_salvage` | El backend capturó una excepción y guardó artefactos; no es completitud |
| `trajectory_shorter_than_target` | Los pasos reales no alcanzan episodios x horas declarados |
| `no_checkpoint` | No hay un paquete de checkpoint recargable con tensores finitos |

Recuperación: no editar el intento. Relanzar la etapa crea `try_<n+1>` desde cero con la misma identidad de tratamiento.

In [ ]:
# ── 7.5  Conteo de motivos de rechazo y cola del ultimo log fallido ──────────────
from collections import Counter

reasons, last_failed = Counter(), None
for stage in ('pilot', 'hpo', 'confirm', 'sensitivity'):
    for treatment, attempt in stage_attempts(stage):
        if not attempt.get('training_complete'):
            reasons.update(attempt.get('reasons', []))
            last_failed = ARTIFACT_ROOT / attempt['path']
if not reasons:
    print('Sin intentos rechazados (o ninguna etapa ejecutada todavia).')
else:
    for reason, count in reasons.most_common():
        print(f'{count:4d}  {reason}')
    log = last_failed / 'child.log'
    if log.is_file():
        print(f'\nUltimas lineas de {log}:')
        print('\n'.join(log.read_text(encoding='utf-8', errors='replace').splitlines()[-30:]))

### 7.7 Monitor de recursos del sistema — RAM / VRAM / GPU / CPU

Instantánea de uso de recursos durante o después de una etapa. Solo imprime; no escribe archivos dentro de la campaña.

In [ ]:
# ── 7.7  Monitor de recursos: RAM / VRAM / CPU / GPU ────────────────────────
import subprocess
import json
import os
import csv
import time
from pathlib import Path
from datetime import datetime

# psutil para RAM y CPU
try:
    import psutil
    ram = psutil.virtual_memory()
    cpu_pct = psutil.cpu_percent(interval=1)
    ram_used_gib  = ram.used / 1024**3
    ram_total_gib = ram.total / 1024**3
    ram_pct = ram.percent
except ImportError:
    ram_used_gib = ram_total_gib = ram_pct = cpu_pct = None
    print("[INFO] psutil no disponible. Instala con: pip install psutil")

# GPU via nvidia-smi
gpu_info = {}
try:
    res = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=10,
    )
    if res.returncode == 0:
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 6:
            gpu_info = {
                "gpu_index"      : parts[0],
                "gpu_name"       : parts[1],
                "gpu_util_pct"   : float(parts[2]),
                "vram_used_mib"  : float(parts[3]),
                "vram_total_mib" : float(parts[4]),
                "gpu_temp_c"     : float(parts[5]),
                "vram_used_gib"  : float(parts[3]) / 1024,
                "vram_total_gib" : float(parts[4]) / 1024,
                "vram_used_pct"  : 100.0 * float(parts[3]) / max(float(parts[4]), 1),
            }
except Exception as e:
    print(f"[WARN] nvidia-smi error: {e}")

snap = {
    "timestamp"       : datetime.now().isoformat(),
    "ram_used_gib"    : round(ram_used_gib or 0, 2),
    "ram_total_gib"   : round(ram_total_gib or 0, 2),
    "ram_used_pct"    : round(ram_pct or 0, 1),
    "cpu_used_pct"    : round(cpu_pct or 0, 1),
    **{k: round(v, 2) if isinstance(v, float) else v for k, v in gpu_info.items()},
}

print(f"{'=' * 55}")
print(f"  SNAPSHOT DE RECURSOS — {snap['timestamp'][:19]}")
print(f"{'=' * 55}")
print(f"  RAM         : {snap['ram_used_gib']:.1f} / {snap['ram_total_gib']:.1f} GiB  ({snap['ram_used_pct']:.0f}%)")
print(f"  CPU         : {snap['cpu_used_pct']:.0f}% utilizado")
if gpu_info:
    print(f"  GPU         : {gpu_info.get('gpu_name', '?')}")
    print(f"  GPU util    : {gpu_info.get('gpu_util_pct', 0):.0f}%")
    print(f"  VRAM usada  : {gpu_info.get('vram_used_gib', 0):.1f} / {gpu_info.get('vram_total_gib', 0):.1f} GiB  ({gpu_info.get('vram_used_pct', 0):.0f}%)")
    print(f"  Temp GPU    : {gpu_info.get('gpu_temp_c', 0):.0f} C")
print("  (instantanea no persistida)")

## Sección 8: KPIs de la campaña nueva

Fuente de verdad: `configs/evaluation/kpis.yaml`, calculado por `uc3m/application/evaluation/kpi_contract.py` desde trazas físicas por paso del simulador. Los escalares agregados de un `results.json` nativo nunca son entrada.

| Eje | KPIs primarios | Unidad |
|---|---|---|
| **E1 / OE.1** Flexibilidad | `district_daily_peak`, `district_ramping`, `one_minus_load_factor` | kW, kW/h, 1 |
| **E2 / OE.2** Emisiones | `co2_total` | kgCO2 |
| **E3 / OE.3** Costos | `energy_cost_total` | USD |

Los diagnósticos (p. ej. `import_energy`, `export_energy`) se reportan pero no se apilan como réplicas estadísticas. La evaluación congelada compara cada política entrenada con la línea base RBC (`uc3m/application/evaluation/baselines.py`) en `test_2025`; `validation_2024` solo se usa para seleccionar hiperparámetros.

> Hasta que una etapa termine con modelos verificados, esta sección solo muestra el catálogo.

In [ ]:
# ── 8.1  Catalogo de KPIs y estado de la evaluacion ─────────────────────────────
from pathlib import Path

import pandas as pd
import yaml

_root = Path(globals().get('PROJECT_ROOT', globals().get('PROJECT_DIR', Path.cwd())))
catalog = yaml.safe_load((_root / 'configs/evaluation/kpis.yaml').read_text(encoding='utf-8'))
frame = pd.DataFrame(catalog['definitions'])
columns = [c for c in ('name', 'axis', 'unit', 'sense', 'level', 'window', 'aggregation', 'primary') if c in frame.columns]
print('moneda:', catalog.get('currency'))
try:
    display(frame[columns])
except NameError:
    print(frame[columns].to_string())

_artifacts = _root / 'outputs/campaigns/iquitos_v1'
_completed = [t for stage in ('confirm', 'sensitivity') for t, a in stage_attempts(stage) if a.get('training_complete')] \
    if 'stage_attempts' in globals() else []
if not _completed:
    print('\nSin intentos completos de confirmacion/sensibilidad: no hay KPIs de politicas entrenadas que mostrar.')
else:
    print(f'\n{len(_completed)} tratamientos completos; la evaluacion congelada en test_2025 produce sus KPIs.')

In [ ]:
# ── 8.2  Curvas de aprendizaje de los intentos de la campaña ───────────────────
# Solo lee timeseries.csv dentro de intentos de outputs/campaigns/iquitos_v1/.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

_root = Path(globals().get('PROJECT_ROOT', globals().get('PROJECT_DIR', Path.cwd())))
_artifacts = _root / 'outputs/campaigns/iquitos_v1'
series = sorted(_artifacts.glob('*/*/*/*/seed_*/try_*/**/timeseries.csv')) if _artifacts.is_dir() else []
if not series:
    print('Sin intentos entrenados en esta campaña: no hay curvas de aprendizaje.')
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    for path in series:
        frame = pd.read_csv(path)
        if {'episode', 'reward_mean'} <= set(frame.columns):
            curve = frame.groupby('episode')['reward_mean'].mean()
            label = '/'.join(path.relative_to(_artifacts).parts[:5])
            ax.plot(curve.index, curve.values, lw=1.5, alpha=0.8, label=label)
    ax.set_xlabel('Episodio')
    ax.set_ylabel('Reward medio por episodio')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

## Sección 9: Evaluación estadística — protocolo congelado

Protocolo de `uc3m/application/evaluation/fresh_analysis.py` (`FrozenProtocol.default()`):

1. **Unidad de réplica** — la semilla de entrenamiento independiente (12 por celda, 1001-1012). Episodios o KPIs distintos no son réplicas.
2. **Endpoints** — E1 → flexibilidad, E2 → CO₂, E3 → costo; ganancia frente a RBC.
3. **Incertidumbre** — IC 95 % bootstrap por semilla, 10 000 remuestreos, semilla 7001; media e IQM.
4. **Pruebas** — familias separadas (omnibus, pares entre algoritmos, impacto frente a RBC) con corrección de Holm.
5. **Cobertura** — un tratamiento faltante bloquea la aceptación; un estudio negativo completo es aceptable.
6. **Limitaciones** — n = 12 semillas por celda: potencia limitada; ausencia de significancia no prueba equivalencia.

In [ ]:
# ── 9.1  Protocolo estadistico congelado y cobertura de la campaña ─────────────
from dataclasses import asdict

from uc3m.application.evaluation.fresh_analysis import FrozenProtocol

protocol = FrozenProtocol.default()
for key, value in asdict(protocol).items():
    print(f'{key:16s} = {value}')

_report = ARTIFACT_ROOT / 'reports' / 'scientific_report.json'
if _report.is_file():
    print('\nReporte cientifico de la campaña:', _report)
    print(json.dumps(json.loads(_report.read_text(encoding='utf-8')).get('decisions', {}), indent=2))
else:
    print('\nSin reporte cientifico: la tabla por semilla requiere la confirmacion completa y su evaluacion congelada.')

### 9.3 Decisión por nivel distrito

La selección no fuerza un ganador: `select_candidate` puede devolver un candidato, `no_unique_candidate` o `infeasible`. La aceptación científica exige confirmación, sensibilidad y evaluación congeladas completas; la aceptación de software y la científica se reportan por separado con `verify_fresh_acceptance`.

In [ ]:
# Verifica la aceptacion de la campaña nueva con la puerta oficial (solo lectura).
from uc3m.application.training.fresh_acceptance import verify_fresh_acceptance

if not MANIFEST_PATH.is_file():
    # Sin campaña preparada no hay nada que aceptar.
    print('Campaña no preparada: no hay modelos entrenados ni aceptacion que verificar.')
else:
    # La puerta revisa identidad, hashes, intentos completos, checkpoints y reporte cientifico.
    report = verify_fresh_acceptance(MANIFEST_PATH)
    print(json.dumps(report.as_dict(), indent=2))
    # not_ready_no_model significa que todavia no hay ningun modelo entrenado y verificado.
    if report.status == 'not_ready_no_model':
        print('Todavia no hay modelos entrenados y verificados en esta campaña.')

## Sección 9b: Hipervolumen y TOPSIS con referencia congelada

La comparación multiobjetivo usa una referencia de hipervolumen común y pesos TOPSIS fijados **antes** de ver la prueba (`FrozenProtocol.topsis_weights`). No se ajustan pesos ni referencias a posteriori, y no se usa una matriz ilustrativa como resultado.

In [ ]:
# ── 9b.1  Referencia de hipervolumen y pesos TOPSIS congelados ──────────────────
from uc3m.application.evaluation import fresh_analysis

print('Referencia de hipervolumen :', fresh_analysis.HV_REFERENCE)
print('Pesos TOPSIS congelados    :', fresh_analysis.FrozenProtocol.default().topsis_weights)
print('Se aplican solo a la tabla por semilla de la campaña; sin ella no hay ranking que mostrar.')

## Sección 10: Resumen final de la sesión

Imprime la identidad de la campaña, el estado de cada etapa y el estado de aceptación. Ejecutar al cierre de la sesión.

In [ ]:
# ── 10.  Resumen final de la sesion ───────────────────────────────────────────
from uc3m.application.training.fresh_acceptance import verify_fresh_acceptance

print('=' * 64)
print('  RESUMEN — campaña', CONFIG['campaign_id'] if 'CONFIG' in globals() else 'fresh_iquitos_20261101_v1')
print('=' * 64)
print('ARTIFACT_ROOT :', ARTIFACT_ROOT)
if not MANIFEST_PATH.is_file():
    print('Estado        : campaña no preparada; nada entrenado.')
else:
    for stage in ('pilot', 'hpo', 'confirm', 'sensitivity'):
        rows = stage_attempts(stage)
        done = sum(1 for _, attempt in rows if attempt.get('training_complete'))
        print(f'{stage:12s}: {done} intentos completos de {len(rows)} intentos' if rows else f'{stage:12s}: sin ejecutar')
    report = verify_fresh_acceptance(MANIFEST_PATH)
    print('Aceptacion    :', report.status, '| software =', report.software_acceptance,
          '| cientifica =', report.scientific_acceptance)

## Informe Técnico de Supervisión — vía nueva

> La celda siguiente verifica, en solo lectura, que el cuaderno y el repositorio usan la única vía de lanzamiento: CLI presente, configuración de campaña presente, lanzadores retirados ausentes y estado real de la campaña.

In [ ]:
# ── INFORME TECNICO DE SUPERVISION (solo lectura) ───────────────────────────────
import json
from pathlib import Path

import yaml

_root = Path(globals().get('PROJECT_ROOT', globals().get('PROJECT_DIR', Path.cwd())))
layout = yaml.safe_load((_root / 'governance/repository_layout.yaml').read_text(encoding='utf-8'))
launch = layout['launch']

checks = {
    'cli_canonica': launch['canonical'] == 'uc3m.application.training.cli'
                    and (_root / 'uc3m/application/training/cli.py').is_file(),
    'config_campaña': (_root / 'configs/experiments/iquitos.yaml').is_file(),
    'dataset_aceptado': json.loads((_root / 'data/campaigns/iquitos_v1/dataset_acceptance.json')
                                   .read_text(encoding='utf-8'))['dataset_verified'] is True,
    'lanzadores_retirados_ausentes': not [p for p in launch['retired'] if (_root / p).exists()],
}
for name, ok in checks.items():
    print(f"[{'OK' if ok else 'FALLA'}] {name}")

_manifest = _root / 'outputs/campaigns/iquitos_v1/campaign_manifest.json'
print('\nCampaña preparada:', _manifest.is_file())
if _manifest.is_file():
    from uc3m.application.training.fresh_acceptance import verify_fresh_acceptance
    print(json.dumps(verify_fresh_acceptance(_manifest).as_dict(), indent=2))
else:
    print('No hay entrenamiento ni aceptacion que reportar todavia.')

## Próximos pasos y referencias

### Entrenamiento
1. Preparar la campaña (**7.1**) y revisar el presupuesto (**6.1**).
2. Autorizar y ejecutar el piloto; medir RAM/VRAM/tiempo antes de pedir recursos para etapas largas.
3. HPO → congelado de configuración → confirmación → sensibilidad, en ese orden, con la misma CLI.
4. Si Colab se desconecta: relanzar la etapa en **7.2**; se crea un intento nuevo desde cero.

### Artefactos
```
outputs/campaigns/iquitos_v1/
  campaign_manifest.json · campaign.json
  stages/<etapa>.json · stages/<etapa>_result.json
  <etapa>/<algoritmo>/<escenario>/<candidato>/seed_<s>/try_<n>/
  reports/scientific_report.json        (solo tras la evaluacion congelada)
```

### Análisis
- KPIs: `configs/evaluation/kpis.yaml` desde trazas físicas; evaluación congelada en `test_2025` frente a RBC.
- Estadística por semilla: IC 95 % bootstrap, Holm, hipervolumen y TOPSIS con referencias congeladas.
- Entrega: `tools/eval/report_fresh_campaign.py` exporta manifiestos, tablas y hashes de la campaña.
- Guía operativa: `docs/architecture/runbook.md`; plan: `docs/plans/entrenamiento.md`.

### Repositorio
[Mac-Tapia/CityLearn](https://github.com/Mac-Tapia/CityLearn) · [MADRLCitytleranflexresdr](https://github.com/Mac-Tapia/MADRLCitytleranflexresdr)
Tesis: *Diseño y validación de un sistema eléctrico inteligente con control multiagente MADRL, Iquitos 2026*
Contacto: mac.tapia@unmsm.edu.pe